# North Star Analytics CRM — Launcher (GitHub Edition)

**Daily operational interface.** Run the setup cells in order in Google Colab, then use the **single CRM launcher at the bottom**. The Master Notebook documents initial infrastructure and development.

This public copy uses placeholders for account-specific paths and spreadsheet identifiers, and has no saved outputs. **Do not use Run All against a live client workbook without reviewing configuration and write operations.** Some functions create or update Google Sheets records when you use the forms. The notebook has been statically checked but not end-to-end tested against Google services.


In [ ]:
# STEP 001 — Install dependencies and import libraries

# Original section: CENTRALIZED LIBRARY IMPORTS
# Google Colab is required for google.colab authentication and Drive mounting.
# Run this cell first in Google Colab. It installs dependencies and imports libraries.
# Installation may take a moment; rerun this cell after restarting the runtime.

%pip -q install gspread google-auth google-api-python-client pandas numpy ipywidgets python-docx openpyxl

import inspect
import json
import re
import shutil
from datetime import date, datetime, timedelta
from decimal import Decimal, InvalidOperation
from html import escape
from pathlib import Path
from uuid import uuid4
from zoneinfo import ZoneInfo

import gspread
import ipywidgets as widgets
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import clear_output, display
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from google.auth import default
from google.colab import auth, drive
from gspread.utils import rowcol_to_a1

CRM_TIMEZONE = ZoneInfo("America/Chicago")
NAVY = RGBColor(18, 35, 65)
COBALT = RGBColor(37, 99, 235)


In [ ]:
# STEP 002 — Connect Google Drive and Google Sheets 

drive.mount("/content/drive")

CRM_ROOT = Path(
    "INSERT DRIVE DIRECTORY HERE"
)
CRM_SPREADSHEET_ID = "INSERT SPREADSHEET ID HERE"

auth.authenticate_user()
credentials, _ = default()
gc = gspread.authorize(credentials)
spreadsheet = gc.open_by_key(CRM_SPREADSHEET_ID)

worksheet_map = {
    "companies_sheet": "Companies",
    "contacts_sheet": "Contacts",
    "activities_sheet": "Activities",
    "opportunities_sheet": "Opportunities",
    "followups_sheet": "Follow-Ups",
    "discovery_sheet": "Discovery",
    "proposals_sheet": "Proposals",
    "projects_sheet": "Projects",
    "project_tasks_sheet": "Project Tasks",
    "continuing_sheet": "Continuing Services",
    "financials_sheet": "Financials",
    "dashboard_sheet": "Dashboard",
    "outreach_goals_sheet": "Outreach Goals",
    "question_library_sheet": "Discovery Question Library",
    "interview_sheet": "Discovery Interviews",
    "scope_sheet": "Proposal Scope Drafts",
    "review_sheet": "Proposal Reviews",
    "template_sheet": "Communication Templates"
}

for variable, sheet_name in worksheet_map.items():
    globals()[variable] = spreadsheet.worksheet(sheet_name)

print("North Star Analytics CRM connected.")


In [ ]:
# STEP 003 — Define CRM schema, stages, and field options

crm_schema = {
    "Dashboard": [
        "Metric", "Value", "Last Updated"
    ],
    "Companies": [
        "Company ID", "Company Name", "Industry", "Business Type",
        "Website", "Phone", "Email", "Address", "City", "State",
        "Number of Locations", "Company Size", "Lead Source",
        "Priority", "Tags", "Date Identified", "Notes"
    ],
    "Contacts": [
        "Contact ID", "Company ID", "First Name", "Last Name",
        "Job Title", "Department", "Email", "Phone",
        "Decision-Maker", "Preferred Contact Method",
        "Tags", "Last Contact Date", "Notes"
    ],
    "Activities": [
        "Activity ID", "Company ID", "Contact ID",
        "Activity Date", "Activity Type", "Direction",
        "Outcome", "Notes", "Next Action", "Follow-Up Date"
    ],
    "Opportunities": [
        "Opportunity ID", "Company ID", "Contact ID",
        "Opportunity Name", "Service Interest", "Pipeline Stage",
        "Outcome", "Estimated Value", "Probability",
        "Weighted Value", "Opportunity Score",
        "Date Opened", "Expected Close Date", "Actual Close Date",
        "Lost Reason", "Tags", "Notes"
    ],
    "Follow-Ups": [
        "Task ID", "Company ID", "Contact ID", "Opportunity ID",
        "Task Description", "Task Type", "Priority",
        "Due Date", "Status", "Completed Date", "Notes"
    ],
    "Discovery": [
        "Discovery ID", "Company ID", "Contact ID",
        "Discovery Date", "Department", "Current Process",
        "Pain Point", "Business Impact", "Current Software",
        "Available Data", "Current KPI", "Desired Outcome",
        "Decision-Maker", "Budget", "Timeline",
        "Recommended Service", "Notes"
    ],
    "Proposals": [
        "Proposal ID", "Opportunity ID", "Company ID",
        "Proposal Date", "Service Package", "Base Price",
        "Add-On Price", "Total Price", "Scope",
        "Exclusions", "Status", "Decision Date",
        "Proposal Link", "Notes"
    ],
    "Projects": [
        "Project ID", "Company ID", "Opportunity ID",
        "Project Name", "Service Type", "Start Date",
        "Target Completion Date", "Actual Completion Date",
        "Project Status", "Contract Value", "Hours Estimated",
        "Hours Worked", "Deliverables", "Project Folder Link",
        "Notes"
    ],
    "Project Tasks": [
        "Project Task ID", "Project ID", "Task Name",
        "Description", "Priority", "Start Date", "Due Date",
        "Status", "Completed Date", "Estimated Hours",
        "Actual Hours", "Notes"
    ],
    "Continuing Services": [
        "Service ID", "Company ID", "Project ID",
        "Service Type", "Monthly Fee", "Billing Frequency",
        "Start Date", "Renewal Date", "Status",
        "Next Review Date", "Deliverables", "Notes"
    ],
    "Financials": [
        "Transaction ID", "Company ID", "Project ID",
        "Transaction Date", "Transaction Type", "Category",
        "Description", "Amount", "Invoice Number",
        "Invoice Due Date", "Payment Date", "Payment Status",
        "Notes"
    ],
    "Tags": [
        "Tag ID", "Tag Category", "Tag Name",
        "Description", "Active"
    ],
    "Settings": [
        "Setting Category", "Setting Name",
        "Setting Value", "Description"
    ]
}

tag_categories = {
    "Business Type": [
        "Automotive Dealership", "Dealer Group",
        "Independent Business", "Multi-Location",
        "Service Business", "Retail", "Nonprofit"
    ],
    "Decision-Maker": [
        "Owner", "General Manager", "Dealer Principal",
        "Fixed Operations Director", "Sales Manager",
        "Finance Director", "Operations Manager",
        "Marketing Director", "Gatekeeper"
    ],
    "Business Needs": [
        "Process Bottlenecks", "Sales Performance",
        "Employee Productivity", "KPI Reporting",
        "Data Quality", "Workflow Automation",
        "Customer Retention", "Inventory Management",
        "Training Gaps", "Dashboard Development"
    ],
    "Sales Priority": [
        "Hot Lead", "Warm Lead", "Cold Lead",
        "High-Value Opportunity", "Budget Confirmed",
        "Decision-Maker Engaged", "Needs Follow-Up",
        "Not Ready", "Referral Source"
    ],
    "Services": [
        "Business Performance Checkup",
        "Process Improvement",
        "Monthly KPI Reporting",
        "Dashboard Maintenance",
        "Data Quality Monitoring",
        "Employee Training",
        "Implementation Support",
        "Full Partnership",
        "Quarterly Review"
    ]
}

pipeline_stages = [
    "Target identified",
    "First contact attempted",
    "Stakeholder identified",
    "Conversation scheduled",
    "Discovery completed",
    "Qualified opportunity",
    "Proposal sent",
    "Decision pending",
    "Won",
    "Lost",
    "Nurture",
    "Project active",
    "Continuing-service opportunity"
]

activity_types = [
    "Phone Call", "Email", "LinkedIn",
    "In-Person Visit", "Video Meeting",
    "Referral", "Follow-Up", "Other"
]

task_statuses = [
    "Not Started", "In Progress",
    "Waiting", "Completed", "Cancelled"
]

scoring_criteria = {
    "Business Need": 25,
    "Decision-Maker Access": 20,
    "Budget Readiness": 20,
    "Timeline Urgency": 15,
    "Engagement Level": 10,
    "Service Fit": 10
}

dropdown_config = {
    "Companies": {
        "Industry": [
            "Automotive", "Retail", "Professional Services",
            "Healthcare", "Manufacturing", "Hospitality",
            "Nonprofit", "Other"
        ],
        "Priority": ["High", "Medium", "Low"]
    },
    "Contacts": {
        "Decision-Maker": ["Yes", "No", "Influencer", "Unknown"],
        "Preferred Contact Method": [
            "Phone", "Email", "LinkedIn", "In Person"
        ]
    },
    "Activities": {
        "Activity Type": activity_types,
        "Direction": ["Outbound", "Inbound"],
        "Outcome": [
            "No Response", "Connected", "Meeting Scheduled",
            "Information Sent", "Follow-Up Needed",
            "Not Interested", "Other"
        ]
    },
    "Opportunities": {
        "Pipeline Stage": pipeline_stages,
        "Outcome": ["Open", "Won", "Lost", "Nurture"]
    },
    "Follow-Ups": {
        "Priority": ["High", "Medium", "Low"],
        "Status": task_statuses
    },
    "Proposals": {
        "Status": [
            "Draft", "Sent", "Under Review",
            "Accepted", "Declined", "Expired"
        ]
    },
    "Projects": {
        "Project Status": [
            "Not Started", "Active", "On Hold",
            "Completed", "Cancelled"
        ]
    },
    "Project Tasks": {
        "Priority": ["High", "Medium", "Low"],
        "Status": task_statuses
    },
    "Continuing Services": {
        "Billing Frequency": [
            "Monthly", "Quarterly", "Annually", "One-Time"
        ],
        "Status": [
            "Proposed", "Active", "Paused",
            "Completed", "Cancelled"
        ]
    },
    "Financials": {
        "Transaction Type": ["Income", "Expense"],
        "Payment Status": [
            "Not Invoiced", "Unpaid", "Partially Paid",
            "Paid", "Overdue", "Cancelled"
        ]
    }
}


In [ ]:
# STEP 004 — Configure opportunity scoring weights

scoring_weights = dict(scoring_criteria)
assert sum(scoring_weights.values()) == 100


In [ ]:
# STEP 005 — Configure consulting service catalog and scope protections

# Original section: RESTORE CONSULTING SERVICE CATALOG
# NORTH STAR ANALYTICS CRM — REPAIRED DAILY LAUNCHER

service_recommendations = {
    "Sales Performance Assessment": {
        "deliverables": [
            "Salesperson performance analysis",
            "Lead conversion comparison",
            "Follow-up behavior assessment",
            "Performance improvement recommendations"
        ],
        "continuing_services": [
            "Monthly KPI Reporting",
            "Employee Training"
        ]
    },
    "Sales Process Improvement": {
        "deliverables": [
            "Current-state sales process map",
            "Sales bottleneck assessment",
            "Future-state workflow recommendations",
            "Implementation roadmap"
        ],
        "continuing_services": [
            "Quarterly Process Review",
            "Implementation Support"
        ]
    },
    "BDC Workflow Assessment": {
        "deliverables": [
            "Lead response-time analysis",
            "Lead assignment assessment",
            "Follow-up workflow map",
            "BDC performance recommendations"
        ],
        "continuing_services": [
            "Monthly KPI Reporting",
            "Data-Quality Monitoring"
        ]
    },
    "Service Workflow Improvement": {
        "deliverables": [
            "Repair order process map",
            "Cycle-time and delay analysis",
            "Workflow improvement recommendations",
            "Implementation priorities"
        ],
        "continuing_services": [
            "Quarterly Process Review",
            "Implementation Support"
        ]
    },
    "Inventory Aging Assessment": {
        "deliverables": [
            "Inventory aging analysis",
            "Slow-moving inventory identification",
            "Inventory turnover assessment",
            "Inventory management recommendations"
        ],
        "continuing_services": [
            "Monthly KPI Reporting",
            "Dashboard Maintenance"
        ]
    },
    "Marketing Performance Dashboard": {
        "deliverables": [
            "Campaign performance analysis",
            "Marketing attribution assessment",
            "Cost-per-sale reporting",
            "Marketing KPI dashboard"
        ],
        "continuing_services": [
            "Monthly KPI Reporting",
            "Dashboard Maintenance"
        ]
    },
    "Data Quality Assessment": {
        "deliverables": [
            "Data source inventory",
            "Missing and inconsistent data assessment",
            "Data validation recommendations",
            "Data quality improvement plan"
        ],
        "continuing_services": [
            "Data-Quality Monitoring",
            "Quarterly Process Review"
        ]
    },
    "Customer Experience Assessment": {
        "deliverables": [
            "Customer feedback analysis",
            "Complaint trend assessment",
            "Customer journey review",
            "Customer experience recommendations"
        ],
        "continuing_services": [
            "Monthly KPI Reporting",
            "Quarterly Process Review"
        ]
    }
}

default_deliverables = [
    "Discovery findings summary",
    "Relevant data assessment",
    "Current performance analysis",
    "Improvement recommendations",
    "Implementation considerations"
]

scope_protection_flags = [
    "Additional Locations",
    "More Than Four Workflows",
    "Major Dashboard Development",
    "Extensive Historical Data Cleanup",
    "Custom Software Development",
    "On-Site Travel",
    "Additional Training Sessions",
    "Urgent Work Outside Established Schedule"
]

print("✓ Consulting service catalog restored.")
print("✓ Services:", len(service_recommendations))
print("✓ Default deliverables:", len(default_deliverables))
print("✓ Scope protections:", len(scope_protection_flags))

print("\nDEPENDENCY CHECK")
for name in [
    "service_recommendations",
    "default_deliverables",
    "scope_protection_flags",
    "generate_consulting_recommendations",
    "create_proposal_scope_draft"
]:
    print(
        f"{'✓ LOADED' if name in globals() else '✗ MISSING'}: {name}"
    )

print("\n✓ No Google Sheets records modified.")


In [ ]:
# STEP 006 — Configure proposal document folder

# Replace the root path in STEP 002 before running.
PROPOSAL_FOLDER = CRM_ROOT / "04_Proposals"
if not PROPOSAL_FOLDER.is_dir():
    raise FileNotFoundError(
        "Proposal directory not found. Configure CRM_ROOT and create 04_Proposals."
    )
print("Proposal document directory configured.")


In [ ]:
# STEP 007 — Core function: create_company

# CRM functions from original cell 8

def create_company(
    company_name,
    industry="Automotive",
    business_type="Automotive Dealership",
    website="",
    phone="",
    email="",
    address="",
    city="",
    state="",
    number_of_locations=1,
    company_size="",
    lead_source="",
    priority="Medium",
    tags=None,
    notes=""
):
    """
    Create a company record in the North Star Analytics CRM.
    Automatically generates a unique company ID.
    """

    if not company_name.strip():
        raise ValueError("Company name is required.")

    company_id = "NSA-C-" + uuid4().hex[:12].upper()

    date_identified = datetime.now().strftime("%Y-%m-%d")

    if tags is None:
        tags = []

    company_record = [
        company_id,
        company_name,
        industry,
        business_type,
        website,
        phone,
        email,
        address,
        city,
        state,
        number_of_locations,
        company_size,
        lead_source,
        priority,
        ", ".join(tags),
        date_identified,
        notes
    ]

    companies_sheet.append_row(
        company_record,
        value_input_option="USER_ENTERED"
    )

    print("✓ Company successfully added!")
    print("Company:", company_name)
    print("Company ID:", company_id)

    return company_id


In [ ]:
# STEP 008 — Core function: create_contact

# CRM functions from original cell 9

def create_contact(
    company_id,
    first_name,
    last_name,
    job_title="",
    department="",
    email="",
    phone="",
    decision_maker="Unknown",
    preferred_contact_method="Email",
    tags=None,
    notes=""
):
    """
    Creates a contact and links it to an existing company.
    Prevents duplicate contacts within the same company.
    """

    # Verify company exists
    company_records = companies_sheet.get_all_records()

    company_exists = any(
        str(company["Company ID"]) == str(company_id)
        for company in company_records
    )

    if not company_exists:
        raise ValueError(
            f"Company ID {company_id} does not exist."
        )

    if not first_name.strip() or not last_name.strip():
        raise ValueError("First and last name are required.")

    # Check for existing contact
    existing_contacts = contacts_sheet.get_all_records()

    for contact in existing_contacts:
        same_company = (
            str(contact["Company ID"]) == str(company_id)
        )

        same_name = (
            str(contact["First Name"]).strip().lower()
            == first_name.strip().lower()
            and
            str(contact["Last Name"]).strip().lower()
            == last_name.strip().lower()
        )

        if same_company and same_name:
            print("⚠ Contact already exists.")
            print("Contact ID:", contact["Contact ID"])
            return contact["Contact ID"]

    # Generate unique contact ID
    contact_id = "NSA-CT-" + uuid4().hex[:12].upper()

    # Prepare contact record
    contact_record = [
        contact_id,
        company_id,
        first_name.strip(),
        last_name.strip(),
        job_title,
        department,
        email,
        phone,
        decision_maker,
        preferred_contact_method,
        ", ".join(tags or []),
        "",
        notes
    ]

    # Save to Google Sheets
    contacts_sheet.append_row(
        contact_record,
        value_input_option="USER_ENTERED"
    )

    print("✓ Contact successfully created!")
    print("Name:", first_name, last_name)
    print("Company ID:", company_id)
    print("Contact ID:", contact_id)

    return contact_id


In [ ]:
# STEP 009 — Core function: log_activity

# CRM functions from original cell 10

def log_activity(
    company_id,
    activity_type,
    outcome,
    contact_id="",
    direction="Outbound",
    notes="",
    next_action="",
    follow_up_date="",
    activity_date=None
):
    """
    Logs outreach activity and updates the contact's
    last contact date when a contact ID is provided.
    """

    # Verify company exists
    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    # Verify contact belongs to the selected company
    if contact_id:
        contacts = contacts_sheet.get_all_records()

        valid_contact = any(
            str(c["Contact ID"]) == str(contact_id)
            and str(c["Company ID"]) == str(company_id)
            for c in contacts
        )

        if not valid_contact:
            raise ValueError(
                "Contact ID does not belong to this company."
            )

    # Set activity date
    if activity_date is None:
        activity_date = date.today().isoformat()

    activity_id = "NSA-A-" + uuid4().hex[:12].upper()

    activity_record = [
        activity_id,
        company_id,
        contact_id,
        activity_date,
        activity_type,
        direction,
        outcome,
        notes,
        next_action,
        follow_up_date
    ]

    # Save activity
    activities_sheet.append_row(
        activity_record,
        value_input_option="USER_ENTERED"
    )

    # Update contact's Last Contact Date
    if contact_id:
        contact_records = contacts_sheet.get_all_records()

        for row_number, contact in enumerate(
            contact_records, start=2
        ):
            if str(contact["Contact ID"]) == str(contact_id):
                contacts_sheet.update_cell(
                    row_number,
                    12,
                    activity_date
                )
                break

    print("✓ Outreach activity recorded!")
    print("Activity ID:", activity_id)
    print("Activity Type:", activity_type)
    print("Outcome:", outcome)

    if follow_up_date:
        print("Follow-up date:", follow_up_date)

    return activity_id


In [ ]:
# STEP 010 — Core function: crm_today / create_followup

# CRM functions from original cell 11

def crm_today():
    """Return today's date in Central Time."""
    return datetime.now(CRM_TIMEZONE).date()

def create_followup(
    company_id,
    task_description,
    due_date,
    contact_id="",
    opportunity_id="",
    task_type="Follow-Up",
    priority="Medium",
    notes=""
):
    """
    Create a follow-up task linked to a company.
    """

    # Verify company exists
    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    # Validate due date
    due_date = date.fromisoformat(str(due_date))

    # Generate task ID
    task_id = "NSA-T-" + uuid4().hex[:12].upper()

    task_record = [
        task_id,
        company_id,
        contact_id,
        opportunity_id,
        task_description,
        task_type,
        priority,
        due_date.isoformat(),
        "Not Started",
        "",
        notes
    ]

    followups_sheet.append_row(
        task_record,
        value_input_option="RAW"
    )

    print("✓ Follow-up task created!")
    print("Task:", task_description)
    print("Due:", due_date)
    print("Priority:", priority)

    return task_id

def show_followups(days_ahead=7):
    """
    Display overdue and upcoming tasks.
    """

    records = followups_sheet.get_all_records()

    today = crm_today()
    cutoff = today + timedelta(days=days_ahead)

    tasks = []

    for record in records:
        if record["Status"] in ["Completed", "Cancelled"]:
            continue

        if not record["Due Date"]:
            continue

        due = date.fromisoformat(str(record["Due Date"]))

        if due <= cutoff:
            if due < today:
                status = "OVERDUE"
            elif due == today:
                status = "DUE TODAY"
            else:
                status = "UPCOMING"

            tasks.append({
                "Task ID": record["Task ID"],
                "Company ID": record["Company ID"],
                "Task": record["Task Description"],
                "Due Date": due.isoformat(),
                "Priority": record["Priority"],
                "Status": status
            })

    if not tasks:
        print("✓ No outstanding follow-ups!")
        return

    df = pd.DataFrame(tasks)

    priority_order = {
        "High": 0,
        "Medium": 1,
        "Low": 2
    }

    df["Priority Rank"] = df["Priority"].map(priority_order)

    df = df.sort_values(
        ["Due Date", "Priority Rank"]
    ).drop(columns=["Priority Rank"])

    display(df)

def complete_followup(task_id):
    """
    Mark a follow-up task as completed.
    """

    records = followups_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):
        if str(record["Task ID"]) == str(task_id):

            followups_sheet.update(
                range_name=f"I{row_number}:J{row_number}",
                values=[[
                    "Completed",
                    crm_today().isoformat()
                ]]
            )

            print("✓ Follow-up completed:", task_id)
            return

    print("⚠ Task ID not found.")


In [ ]:
# STEP 011 — Core function: create_opportunity / update_opportunity_stage

# CRM functions from original cell 12

def create_opportunity(
    company_id,
    opportunity_name,
    service_interest,
    estimated_value,
    contact_id="",
    pipeline_stage="Target identified",
    tags=None,
    expected_close_date="",
    notes=""
):
    """
    Create a new sales opportunity and calculate
    its weighted pipeline value.
    """

    # Verify company exists
    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    # Verify stage
    if pipeline_stage not in stage_probabilities:
        raise ValueError("Invalid pipeline stage.")

    # Calculate probability and weighted value
    probability = stage_probabilities[pipeline_stage]
    weighted_value = float(estimated_value) * probability / 100

    # Generate unique opportunity ID
    opportunity_id = "NSA-O-" + uuid4().hex[:12].upper()

    # Prepare record
    opportunity_record = [
        opportunity_id,
        company_id,
        contact_id,
        opportunity_name,
        service_interest,
        pipeline_stage,
        "Open",
        float(estimated_value),
        probability,
        weighted_value,
        "",
        crm_today().isoformat(),
        expected_close_date,
        "",
        "",
        ", ".join(tags or []),
        notes
    ]

    opportunities_sheet.append_row(
        opportunity_record,
        value_input_option="RAW"
    )

    print("✓ Opportunity created!")
    print("Opportunity:", opportunity_name)
    print("Estimated Value: $", estimated_value)
    print("Probability:", probability, "%")
    print("Weighted Value: $", weighted_value)

    return opportunity_id

def update_opportunity_stage(opportunity_id, new_stage):
    """
    Update an opportunity's pipeline stage,
    probability, and weighted value.
    """

    if new_stage not in stage_probabilities:
        raise ValueError("Invalid pipeline stage.")

    records = opportunities_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):

        if str(record["Opportunity ID"]) == str(opportunity_id):

            probability = stage_probabilities[new_stage]
            estimated_value = float(record["Estimated Value"])
            weighted_value = estimated_value * probability / 100

            if new_stage == "Won":
                outcome = "Won"
            elif new_stage == "Lost":
                outcome = "Lost"
            elif new_stage == "Nurture":
                outcome = "Nurture"
            else:
                outcome = "Open"

            opportunities_sheet.update(
                range_name=f"F{row_number}:J{row_number}",
                values=[[
                    new_stage,
                    outcome,
                    estimated_value,
                    probability,
                    weighted_value
                ]]
            )

            print("✓ Opportunity updated!")
            print("New Stage:", new_stage)
            print("Weighted Value: $", weighted_value)

            return

    print("⚠ Opportunity not found.")


In [ ]:
# STEP 012 — Core function: score_opportunity / show_priority_opportunities

# CRM functions from original cell 13

def score_opportunity(
    opportunity_id,
    business_need,
    decision_maker_access,
    budget_readiness,
    timeline_urgency,
    engagement_level,
    service_fit
):
    """
    Score a sales opportunity using six weighted criteria.

    Each input must be between 0 and 5.
    Final score ranges from 0 to 100.
    """

    ratings = {
        "Business Need": business_need,
        "Decision-Maker Access": decision_maker_access,
        "Budget Readiness": budget_readiness,
        "Timeline Urgency": timeline_urgency,
        "Engagement Level": engagement_level,
        "Service Fit": service_fit
    }

    # Validate ratings
    for criterion, rating in ratings.items():
        if not isinstance(rating, (int, float)):
            raise ValueError(f"{criterion} must be numeric.")

        if rating < 0 or rating > 5:
            raise ValueError(
                f"{criterion} must be between 0 and 5."
            )

    # Calculate weighted score
    total_score = sum(
        (ratings[criterion] / 5) * scoring_weights[criterion]
        for criterion in ratings
    )

    total_score = round(total_score, 2)

    # Assign priority classification
    if total_score >= 80:
        classification = "High Priority"
    elif total_score >= 60:
        classification = "Qualified Prospect"
    elif total_score >= 40:
        classification = "Developing Opportunity"
    else:
        classification = "Low Priority / Nurture"

    # Find opportunity and update score
    records = opportunities_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):

        if str(record["Opportunity ID"]) == str(opportunity_id):

            opportunities_sheet.update(
                range_name=f"K{row_number}",
                values=[[total_score]]
            )

            print("✓ Opportunity scored successfully!")
            print("Opportunity:", record["Opportunity Name"])
            print("Score:", total_score, "/ 100")
            print("Classification:", classification)

            return {
                "opportunity_id": opportunity_id,
                "score": total_score,
                "classification": classification,
                "ratings": ratings
            }

    raise ValueError("Opportunity ID not found.")

def show_priority_opportunities():
    """
    Display open opportunities ranked by score.
    """

    records = opportunities_sheet.get_all_records()

    active = [
        record for record in records
        if record["Outcome"] == "Open"
    ]

    if not active:
        print("No open opportunities found.")
        return

    df = pd.DataFrame(active)

    df["Opportunity Score"] = pd.to_numeric(
        df["Opportunity Score"],
        errors="coerce"
    )

    df = df.sort_values(
        "Opportunity Score",
        ascending=False,
        na_position="last"
    )

    display(df[[
        "Opportunity ID",
        "Company ID",
        "Opportunity Name",
        "Pipeline Stage",
        "Estimated Value",
        "Opportunity Score"
    ]])


In [ ]:
# STEP 013 — Core function: create_discovery / show_company_discoveries

# CRM functions from original cell 14

def create_discovery(
    company_id,
    department,
    current_process,
    pain_point,
    business_impact,
    contact_id="",
    current_software="",
    available_data="",
    current_kpi="",
    desired_outcome="",
    decision_maker="",
    budget="",
    timeline="",
    recommended_service="",
    notes=""
):
    """
    Create a structured business discovery record.
    Links operational problems to available data,
    KPIs, and potential consulting services.
    """

    # Verify company exists
    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    # Validate department
    if department not in discovery_departments:
        raise ValueError(
            f"Invalid department. Choose from: {discovery_departments}"
        )

    # Verify contact belongs to company
    if contact_id:
        contacts = contacts_sheet.get_all_records()

        valid_contact = any(
            str(c["Contact ID"]) == str(contact_id)
            and str(c["Company ID"]) == str(company_id)
            for c in contacts
        )

        if not valid_contact:
            raise ValueError(
                "Contact does not belong to this company."
            )

    # Generate discovery ID
    discovery_id = "NSA-D-" + uuid4().hex[:12].upper()

    discovery_record = [
        discovery_id,
        company_id,
        contact_id,
        crm_today().isoformat(),
        department,
        current_process,
        pain_point,
        business_impact,
        current_software,
        available_data,
        current_kpi,
        desired_outcome,
        decision_maker,
        budget,
        timeline,
        recommended_service,
        notes
    ]

    discovery_sheet.append_row(
        discovery_record,
        value_input_option="RAW"
    )

    print("✓ Discovery assessment saved!")
    print("Discovery ID:", discovery_id)
    print("Department:", department)
    print("Pain Point:", pain_point)
    print("Recommended Service:", recommended_service)

    return discovery_id

def show_company_discoveries(company_id):
    """
    Retrieve all discovery findings for a company.
    """

    records = discovery_sheet.get_all_records()

    matches = [
        record for record in records
        if str(record["Company ID"]) == str(company_id)
    ]

    if not matches:
        print("No discovery assessments found.")
        return

    df = pd.DataFrame(matches)

    display(df[[
        "Discovery Date",
        "Department",
        "Current Process",
        "Pain Point",
        "Business Impact",
        "Current KPI",
        "Recommended Service"
    ]])


In [ ]:
# STEP 014 — Core function: crm_money / create_proposal

# CRM functions from original cell 15

def crm_money(value):
    """Validate and standardize currency amounts."""
    try:
        amount = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError("Enter a valid monetary amount.")

    if not amount.is_finite() or amount < 0:
        raise ValueError("Amount must be finite and nonnegative.")

    return amount.quantize(Decimal("0.01"))

def create_proposal(
    opportunity_id,
    service_package,
    base_price,
    addon_price=0,
    scope="",
    exclusions="",
    proposal_link="",
    notes=""
):
    """
    Create a proposal linked to an existing opportunity.
    Calculate the total proposed contract value.
    """

    opportunities = opportunities_sheet.get_all_records()

    opportunity = next(
        (
            record for record in opportunities
            if str(record["Opportunity ID"]) == str(opportunity_id)
        ),
        None
    )

    if opportunity is None:
        raise ValueError("Opportunity ID does not exist.")

    if opportunity["Outcome"] in ["Lost", "Won"]:
        raise ValueError(
            "Cannot create a new proposal for a closed opportunity."
        )

    base = crm_money(base_price)
    addons = crm_money(addon_price)
    total = base + addons

    proposal_id = "NSA-P-" + uuid4().hex[:12].upper()

    proposal_record = [
        proposal_id,
        opportunity_id,
        opportunity["Company ID"],
        crm_today().isoformat(),
        service_package,
        float(base),
        float(addons),
        float(total),
        scope,
        exclusions,
        "Draft",
        "",
        proposal_link,
        notes
    ]

    proposals_sheet.append_row(
        proposal_record,
        value_input_option="RAW"
    )

    print("✓ Proposal created successfully!")
    print("Proposal ID:", proposal_id)
    print("Service:", service_package)
    print(f"Base Price: ${base:,.2f}")
    print(f"Add-Ons: ${addons:,.2f}")
    print(f"Total Proposal: ${total:,.2f}")
    print("Status: Draft")

    return proposal_id

def update_proposal_status(proposal_id, new_status):
    """Update proposal status and record decision dates."""

    valid_statuses = [
        "Draft", "Sent", "Under Review",
        "Accepted", "Declined", "Expired"
    ]

    if new_status not in valid_statuses:
        raise ValueError("Invalid proposal status.")

    records = proposals_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):
        if str(record["Proposal ID"]) == str(proposal_id):

            decision_date = (
                crm_today().isoformat()
                if new_status in ["Accepted", "Declined", "Expired"]
                else ""
            )

            proposals_sheet.update(
                range_name=f"K{row_number}:L{row_number}",
                values=[[new_status, decision_date]]
            )

            print("✓ Proposal status updated:", new_status)
            return

    raise ValueError("Proposal ID not found.")

def show_proposals():
    """Display proposals and their current statuses."""

    records = proposals_sheet.get_all_records()

    if not records:
        print("No proposals have been created yet.")
        return

    df = pd.DataFrame(records)

    display(df[[
        "Proposal ID",
        "Company ID",
        "Service Package",
        "Total Price",
        "Status",
        "Decision Date"
    ]])


In [ ]:
# STEP 015 — Core function: create_project / create_project_task

# CRM functions from original cell 16

def create_project(
    company_id,
    project_name,
    service_type,
    start_date,
    target_completion_date,
    contract_value,
    opportunity_id="",
    hours_estimated=0,
    deliverables="",
    project_folder_link="",
    notes=""
):
    """Create a project linked to an existing company."""

    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    if opportunity_id:
        opportunities = opportunities_sheet.get_all_records()

        valid_opportunity = any(
            str(o["Opportunity ID"]) == str(opportunity_id)
            and str(o["Company ID"]) == str(company_id)
            for o in opportunities
        )

        if not valid_opportunity:
            raise ValueError(
                "Opportunity does not belong to this company."
            )

    start = date.fromisoformat(str(start_date))
    target = date.fromisoformat(str(target_completion_date))

    if target < start:
        raise ValueError(
            "Target completion date cannot precede start date."
        )

    value = crm_money(contract_value)

    if hours_estimated < 0:
        raise ValueError("Estimated hours cannot be negative.")

    project_id = "NSA-PR-" + uuid4().hex[:12].upper()

    project_record = [
        project_id,
        company_id,
        opportunity_id,
        project_name,
        service_type,
        start.isoformat(),
        target.isoformat(),
        "",
        "Not Started",
        float(value),
        float(hours_estimated),
        0,
        deliverables,
        project_folder_link,
        notes
    ]

    projects_sheet.append_row(
        project_record,
        value_input_option="RAW"
    )

    print("✓ Project created!")
    print("Project:", project_name)
    print("Project ID:", project_id)
    print(f"Contract Value: ${value:,.2f}")

    return project_id

def create_project_task(
    project_id,
    task_name,
    due_date,
    description="",
    priority="Medium",
    start_date="",
    estimated_hours=0,
    notes=""
):
    """Create a task linked to an existing project."""

    projects = projects_sheet.get_all_records()

    if not any(
        str(p["Project ID"]) == str(project_id)
        for p in projects
    ):
        raise ValueError("Project ID does not exist.")

    due = date.fromisoformat(str(due_date))

    if start_date:
        start = date.fromisoformat(str(start_date))

        if due < start:
            raise ValueError(
                "Due date cannot precede task start date."
            )

    if estimated_hours < 0:
        raise ValueError("Estimated hours cannot be negative.")

    task_id = "NSA-PT-" + uuid4().hex[:12].upper()

    task_record = [
        task_id,
        project_id,
        task_name,
        description,
        priority,
        start_date,
        due.isoformat(),
        "Not Started",
        "",
        float(estimated_hours),
        0,
        notes
    ]

    project_tasks_sheet.append_row(
        task_record,
        value_input_option="RAW"
    )

    print("✓ Project task created!")
    print("Task:", task_name)
    print("Due:", due)

    return task_id

def complete_project_task(task_id, actual_hours=0):
    """Complete a task and record actual hours."""

    if actual_hours < 0:
        raise ValueError("Actual hours cannot be negative.")

    records = project_tasks_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):
        if str(record["Project Task ID"]) == str(task_id):

            project_tasks_sheet.update(
                range_name=f"H{row_number}:K{row_number}",
                values=[[
                    "Completed",
                    crm_today().isoformat(),
                    record["Estimated Hours"],
                    float(actual_hours)
                ]]
            )

            print("✓ Task completed:", task_id)
            return

    raise ValueError("Project task not found.")

def show_project_progress(project_id):
    """Display task progress and hours for a project."""

    projects = projects_sheet.get_all_records()

    project = next(
        (
            p for p in projects
            if str(p["Project ID"]) == str(project_id)
        ),
        None
    )

    if project is None:
        raise ValueError("Project ID not found.")

    tasks = [
        t for t in project_tasks_sheet.get_all_records()
        if str(t["Project ID"]) == str(project_id)
    ]

    total_tasks = len(tasks)
    completed_tasks = sum(
        t["Status"] == "Completed" for t in tasks
    )

    progress = (
        completed_tasks / total_tasks * 100
        if total_tasks else 0
    )

    actual_hours = sum(
        float(t["Actual Hours"] or 0) for t in tasks
    )

    estimated_hours = sum(
        float(t["Estimated Hours"] or 0) for t in tasks
    )

    print("Project:", project["Project Name"])
    print(f"Progress: {progress:.1f}%")
    print(f"Completed Tasks: {completed_tasks}/{total_tasks}")
    print(f"Estimated Task Hours: {estimated_hours:.1f}")
    print(f"Actual Task Hours: {actual_hours:.1f}")

    if tasks:
        display(pd.DataFrame(tasks)[[
            "Task Name", "Priority", "Due Date",
            "Status", "Estimated Hours", "Actual Hours"
        ]])


In [ ]:
# STEP 016 — Core function: create_continuing_service / update_service_status

# CRM functions from original cell 17

def create_continuing_service(
    company_id,
    service_type,
    fee,
    billing_frequency,
    start_date,
    renewal_date="",
    project_id="",
    next_review_date="",
    deliverables="",
    notes=""
):
    """Create a continuing-service agreement."""

    companies = companies_sheet.get_all_records()

    if not any(
        str(c["Company ID"]) == str(company_id)
        for c in companies
    ):
        raise ValueError("Company ID does not exist.")

    if service_type not in service_types:
        raise ValueError("Invalid service type.")

    if billing_frequency not in billing_frequencies:
        raise ValueError("Invalid billing frequency.")

    if project_id:
        projects = projects_sheet.get_all_records()

        if not any(
            str(p["Project ID"]) == str(project_id)
            and str(p["Company ID"]) == str(company_id)
            for p in projects
        ):
            raise ValueError(
                "Project does not belong to this company."
            )

    fee = crm_money(fee)
    start = date.fromisoformat(str(start_date))

    if renewal_date:
        renewal = date.fromisoformat(str(renewal_date))
        if renewal < start:
            raise ValueError("Renewal date precedes start date.")

    if next_review_date:
        date.fromisoformat(str(next_review_date))

    service_id = "NSA-CS-" + uuid4().hex[:12].upper()

    service_record = [
        service_id,
        company_id,
        project_id,
        service_type,
        float(fee),
        billing_frequency,
        start.isoformat(),
        renewal_date,
        "Proposed",
        next_review_date,
        deliverables,
        notes
    ]

    continuing_sheet.append_row(
        service_record,
        value_input_option="RAW"
    )

    print("✓ Continuing-service agreement created!")
    print("Service ID:", service_id)
    print("Service:", service_type)
    print(f"Fee: ${fee:,.2f} ({billing_frequency})")
    print("Status: Proposed")

    return service_id

def update_service_status(service_id, new_status):
    """Update continuing-service agreement status."""

    valid_statuses = [
        "Proposed", "Active", "Paused",
        "Completed", "Cancelled"
    ]

    if new_status not in valid_statuses:
        raise ValueError("Invalid service status.")

    records = continuing_sheet.get_all_records()

    for row_number, record in enumerate(records, start=2):
        if str(record["Service ID"]) == str(service_id):

            continuing_sheet.update(
                range_name=f"I{row_number}",
                values=[[new_status]]
            )

            print("✓ Service status updated:", new_status)
            return

    raise ValueError("Service ID not found.")

def show_recurring_revenue():
    """Calculate active monthly and annual recurring revenue."""

    records = continuing_sheet.get_all_records()

    monthly_revenue = 0

    for record in records:
        if record["Status"] != "Active":
            continue

        fee = float(record["Monthly Fee"] or 0)
        frequency = record["Billing Frequency"]

        if frequency == "Monthly":
            monthly_revenue += fee
        elif frequency == "Quarterly":
            monthly_revenue += fee / 3
        elif frequency == "Annually":
            monthly_revenue += fee / 12

    annual_revenue = monthly_revenue * 12

    print("NORTH STAR ANALYTICS")
    print("Recurring Revenue Summary")
    print("-------------------------")
    print(f"Monthly Recurring Revenue: ${monthly_revenue:,.2f}")
    print(f"Annualized Recurring Revenue: ${annual_revenue:,.2f}")

    return monthly_revenue, annual_revenue

def show_upcoming_renewals(days_ahead=30):
    """Display active agreements approaching renewal."""

    records = continuing_sheet.get_all_records()

    today = crm_today()
    cutoff = today + timedelta(days=days_ahead)

    renewals = []

    for record in records:
        if record["Status"] != "Active":
            continue

        if not record["Renewal Date"]:
            continue

        renewal = date.fromisoformat(str(record["Renewal Date"]))

        if renewal <= cutoff:
            renewals.append({
                "Company ID": record["Company ID"],
                "Service": record["Service Type"],
                "Renewal Date": renewal.isoformat(),
                "Fee": record["Monthly Fee"],
                "Billing Frequency": record["Billing Frequency"]
            })

    if renewals:
        display(
            pd.DataFrame(renewals).sort_values("Renewal Date")
        )
    else:
        print("✓ No renewals due within the selected period.")


In [ ]:
# STEP 017 — Core function: record_financial_transaction / show_financial_summary

# CRM functions from original cell 18

def record_financial_transaction(
    transaction_type,
    category,
    description,
    amount,
    company_id="",
    project_id="",
    invoice_number="",
    invoice_due_date="",
    payment_date="",
    payment_status="",
    notes=""
):
    """Record an invoice, payment, or business expense."""

    valid_types = ["Invoice", "Payment", "Expense"]

    if transaction_type not in valid_types:
        raise ValueError("Invalid transaction type.")

    if category not in financial_categories:
        raise ValueError("Invalid financial category.")

    amount = crm_money(amount)

    if amount <= 0:
        raise ValueError("Transaction amount must be greater than zero.")

    if company_id:
        companies = companies_sheet.get_all_records()

        if not any(
            str(c["Company ID"]) == str(company_id)
            for c in companies
        ):
            raise ValueError("Company ID does not exist.")

    if project_id:
        projects = projects_sheet.get_all_records()

        matching_project = next(
            (
                p for p in projects
                if str(p["Project ID"]) == str(project_id)
            ),
            None
        )

        if matching_project is None:
            raise ValueError("Project ID does not exist.")

        if company_id and str(
            matching_project["Company ID"]
        ) != str(company_id):
            raise ValueError(
                "Project does not belong to the selected company."
            )

    if invoice_due_date:
        date.fromisoformat(str(invoice_due_date))

    if payment_date:
        date.fromisoformat(str(payment_date))

    if transaction_type in ["Invoice", "Payment"]:
        if not company_id:
            raise ValueError(
                "Invoices and payments require a Company ID."
            )
        if not invoice_number:
            raise ValueError(
                "Invoices and payments require an invoice number."
            )

    if transaction_type == "Invoice":
        existing = financials_sheet.get_all_records()

        if any(
            r["Transaction Type"] == "Invoice"
            and str(r["Invoice Number"]) == str(invoice_number)
            for r in existing
        ):
            raise ValueError("Invoice number already exists.")

        payment_status = "Unpaid"

    elif transaction_type == "Payment":
        payment_status = "Received"
        if not payment_date:
            payment_date = crm_today().isoformat()

    else:
        payment_status = "Paid"
        if not payment_date:
            payment_date = crm_today().isoformat()

    transaction_id = "NSA-F-" + uuid4().hex[:12].upper()

    transaction_record = [
        transaction_id,
        company_id,
        project_id,
        crm_today().isoformat(),
        transaction_type,
        category,
        description,
        float(amount),
        invoice_number,
        invoice_due_date,
        payment_date,
        payment_status,
        notes
    ]

    financials_sheet.append_row(
        transaction_record,
        value_input_option="RAW"
    )

    print("✓ Financial transaction recorded!")
    print("Transaction ID:", transaction_id)
    print("Type:", transaction_type)
    print(f"Amount: ${amount:,.2f}")

    return transaction_id

def show_financial_summary():
    """Calculate cash received, expenses, and net cash flow."""

    records = financials_sheet.get_all_records()

    payments = sum(
        float(r["Amount"] or 0)
        for r in records
        if r["Transaction Type"] == "Payment"
    )

    expenses = sum(
        float(r["Amount"] or 0)
        for r in records
        if r["Transaction Type"] == "Expense"
    )

    invoiced = sum(
        float(r["Amount"] or 0)
        for r in records
        if r["Transaction Type"] == "Invoice"
    )

    net_cash_flow = payments - expenses

    print("NORTH STAR ANALYTICS")
    print("Financial Summary")
    print("------------------------------")
    print(f"Total Invoiced: ${invoiced:,.2f}")
    print(f"Payments Received: ${payments:,.2f}")
    print(f"Business Expenses: ${expenses:,.2f}")
    print(f"Net Cash Flow: ${net_cash_flow:,.2f}")

    return {
        "invoiced": invoiced,
        "payments": payments,
        "expenses": expenses,
        "net_cash_flow": net_cash_flow
    }

def show_outstanding_invoices():
    """Calculate invoice balances using recorded payments."""

    records = financials_sheet.get_all_records()

    invoices = [
        r for r in records
        if r["Transaction Type"] == "Invoice"
    ]

    payments = [
        r for r in records
        if r["Transaction Type"] == "Payment"
    ]

    outstanding = []

    for invoice in invoices:
        invoice_number = str(invoice["Invoice Number"])
        company_id = str(invoice["Company ID"])

        paid = sum(
            float(p["Amount"] or 0)
            for p in payments
            if str(p["Invoice Number"]) == invoice_number
            and str(p["Company ID"]) == company_id
        )

        balance = round(
            float(invoice["Amount"] or 0) - paid, 2
        )

        if balance <= 0:
            continue

        due_text = str(invoice["Invoice Due Date"]).strip()
        overdue = False

        if due_text:
            overdue = date.fromisoformat(due_text) < crm_today()

        outstanding.append({
            "Invoice Number": invoice_number,
            "Company ID": company_id,
            "Invoice Amount": float(invoice["Amount"]),
            "Paid": round(paid, 2),
            "Outstanding": balance,
            "Due Date": due_text,
            "Status": "Overdue" if overdue else "Outstanding"
        })

    if not outstanding:
        print("✓ No outstanding invoices.")
        return

    display(pd.DataFrame(outstanding))


In [ ]:
# STEP 018 — Core function: crm_number / refresh_crm_dashboard

# CRM functions from original cell 19

def crm_number(value):
    """Safely convert Google Sheets values into numbers."""
    try:
        return float(str(value).replace("$", "").replace(",", ""))
    except (ValueError, TypeError):
        return 0.0

def refresh_crm_dashboard():
    """
    Read the CRM worksheets and refresh the executive dashboard.
    This function does not modify underlying client records.
    """

    companies = companies_sheet.get_all_records()
    contacts = contacts_sheet.get_all_records()
    activities = activities_sheet.get_all_records()
    opportunities = opportunities_sheet.get_all_records()
    followups = followups_sheet.get_all_records()
    proposals = proposals_sheet.get_all_records()
    projects = projects_sheet.get_all_records()
    services = continuing_sheet.get_all_records()
    financials = financials_sheet.get_all_records()

    today = crm_today()
    next_week = today + timedelta(days=7)

    # Prospecting
    total_companies = len(companies)
    total_contacts = len(contacts)
    total_activities = len(activities)

    # Sales pipeline
    open_opportunities = [
        o for o in opportunities
        if str(o["Outcome"]).strip() not in ["Won", "Lost"]
    ]

    open_pipeline_value = sum(
        crm_number(o["Estimated Value"])
        for o in open_opportunities
    )

    weighted_pipeline_value = sum(
        crm_number(o["Weighted Value"])
        for o in open_opportunities
    )

    proposals_sent = sum(
        str(p["Status"]).strip() in [
            "Sent", "Under Review"
        ]
        for p in proposals
    )

    # Follow-ups
    overdue_tasks = 0
    tasks_due_today = 0
    upcoming_tasks = 0

    for task in followups:
        if str(task["Status"]).strip() in [
            "Completed", "Cancelled"
        ]:
            continue

        due_text = str(task["Due Date"]).strip()

        if not due_text:
            continue

        try:
            due = date.fromisoformat(due_text)
        except ValueError:
            continue

        if due < today:
            overdue_tasks += 1
        elif due == today:
            tasks_due_today += 1
        elif due <= next_week:
            upcoming_tasks += 1

    # Project management
    active_projects = sum(
        str(p["Project Status"]).strip() in [
            "Active", "In Progress"
        ]
        for p in projects
    )

    completed_projects = sum(
        str(p["Project Status"]).strip() == "Completed"
        for p in projects
    )

    # Continuing services
    active_services = [
        s for s in services
        if str(s["Status"]).strip() == "Active"
    ]

    monthly_recurring_revenue = 0.0

    for service in active_services:
        fee = crm_number(service["Monthly Fee"])
        frequency = str(service["Billing Frequency"]).strip()

        if frequency == "Monthly":
            monthly_recurring_revenue += fee
        elif frequency == "Quarterly":
            monthly_recurring_revenue += fee / 3
        elif frequency == "Annually":
            monthly_recurring_revenue += fee / 12

    annualized_recurring_revenue = monthly_recurring_revenue * 12

    # Financial performance
    total_invoiced = sum(
        crm_number(f["Amount"])
        for f in financials
        if f["Transaction Type"] == "Invoice"
    )

    payments_received = sum(
        crm_number(f["Amount"])
        for f in financials
        if f["Transaction Type"] == "Payment"
    )

    business_expenses = sum(
        crm_number(f["Amount"])
        for f in financials
        if f["Transaction Type"] == "Expense"
    )

    net_cash_flow = payments_received - business_expenses

    # Aggregate outstanding balances, including partial payments
    invoice_records = [
        f for f in financials
        if f["Transaction Type"] == "Invoice"
    ]

    payment_records = [
        f for f in financials
        if f["Transaction Type"] == "Payment"
    ]

    outstanding_balance = 0.0

    for invoice in invoice_records:
        paid = sum(
            crm_number(payment["Amount"])
            for payment in payment_records
            if str(payment["Invoice Number"]) ==
               str(invoice["Invoice Number"])
            and str(payment["Company ID"]) ==
                str(invoice["Company ID"])
        )

        outstanding_balance += max(
            0,
            crm_number(invoice["Amount"]) - paid
        )

    # Build dashboard rows
    metrics = [
        ("PROSPECTING", "", ""),
        ("Total Companies", total_companies, ""),
        ("Total Contacts", total_contacts, ""),
        ("Total Activities", total_activities, ""),

        ("SALES PIPELINE", "", ""),
        ("Open Opportunities", len(open_opportunities), ""),
        ("Open Pipeline Value", round(open_pipeline_value, 2), ""),
        ("Weighted Pipeline Value", round(weighted_pipeline_value, 2), ""),
        ("Proposals Awaiting Decision", proposals_sent, ""),

        ("FOLLOW-UPS", "", ""),
        ("Overdue Follow-Ups", overdue_tasks, ""),
        ("Follow-Ups Due Today", tasks_due_today, ""),
        ("Follow-Ups Next 7 Days", upcoming_tasks, ""),

        ("PROJECTS", "", ""),
        ("Active Projects", active_projects, ""),
        ("Completed Projects", completed_projects, ""),

        ("CONTINUING SERVICES", "", ""),
        ("Active Agreements", len(active_services), ""),
        ("Monthly Recurring Revenue", round(monthly_recurring_revenue, 2), ""),
        ("Annualized Recurring Revenue", round(annualized_recurring_revenue, 2), ""),

        ("FINANCIAL PERFORMANCE", "", ""),
        ("Total Invoiced", round(total_invoiced, 2), ""),
        ("Payments Received", round(payments_received, 2), ""),
        ("Business Expenses", round(business_expenses, 2), ""),
        ("Net Cash Flow", round(net_cash_flow, 2), ""),
        ("Outstanding Invoice Balance", round(outstanding_balance, 2), "")
    ]

    timestamp = datetime.now(
        ZoneInfo("America/Chicago")
    ).strftime("%Y-%m-%d %I:%M %p CT")

    rows = [["Metric", "Value", "Last Updated"]]

    for metric, value, _ in metrics:
        rows.append([metric, value, timestamp])

    # Clear previous dashboard values and write fresh results
    dashboard_sheet.clear()

    dashboard_sheet.update(
        range_name="A1",
        values=rows,
        value_input_option="RAW"
    )

    dashboard_sheet.freeze(rows=1)

    print("✓ Executive CRM dashboard refreshed!")
    print("Last Updated:", timestamp)
    print()
    print("Companies:", total_companies)
    print("Open Opportunities:", len(open_opportunities))
    print(f"Pipeline Value: ${open_pipeline_value:,.2f}")
    print("Overdue Follow-Ups:", overdue_tasks)
    print("Active Projects:", active_projects)
    print(f"Monthly Recurring Revenue: ${monthly_recurring_revenue:,.2f}")
    print(f"Payments Received: ${payments_received:,.2f}")
    print(f"Outstanding Invoices: ${outstanding_balance:,.2f}")

    return {
        "companies": total_companies,
        "open_opportunities": len(open_opportunities),
        "pipeline_value": open_pipeline_value,
        "overdue_followups": overdue_tasks,
        "active_projects": active_projects,
        "monthly_recurring_revenue": monthly_recurring_revenue,
        "payments_received": payments_received,
        "outstanding_invoices": outstanding_balance
    }


In [ ]:
# STEP 019 — Core function: daily_sales_command_center

# CRM functions from original cell 21

def daily_sales_command_center(days_ahead=7, top_n=10):
    """
    Generate a daily sales action report from existing CRM data.
    Does not modify underlying CRM records.
    """

    if not isinstance(days_ahead, int) or days_ahead < 0:
        raise ValueError("days_ahead must be a nonnegative integer.")

    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    today = crm_today()
    cutoff = today + timedelta(days=days_ahead)

    companies = companies_sheet.get_all_records()
    contacts = contacts_sheet.get_all_records()
    activities = activities_sheet.get_all_records()
    opportunities = opportunities_sheet.get_all_records()
    followups = followups_sheet.get_all_records()
    proposals = proposals_sheet.get_all_records()

    company_names = {
        str(c["Company ID"]): c["Company Name"]
        for c in companies
    }

    contact_names = {
        str(c["Contact ID"]): (
            str(c["First Name"]) + " " + str(c["Last Name"])
        ).strip()
        for c in contacts
    }

    opportunity_names = {
        str(o["Opportunity ID"]): o["Opportunity Name"]
        for o in opportunities
    }

    def parse_date(value):
        try:
            return date.fromisoformat(str(value).strip())
        except (ValueError, TypeError):
            return None

    def show_section(title, records, columns, limit=None):
        print("\n" + "=" * 65)
        print(title)
        print("=" * 65)

        if not records:
            print("Nothing requiring attention.")
            return

        df = pd.DataFrame(records)

        if limit:
            df = df.head(limit)

        display(df[columns])

    # 1. Overdue and upcoming follow-ups
    overdue = []
    due_today = []
    upcoming = []

    for task in followups:
        if str(task["Status"]).strip() in [
            "Completed", "Cancelled"
        ]:
            continue

        due = parse_date(task["Due Date"])

        if due is None:
            continue

        item = {
            "Task ID": task["Task ID"],
            "Company": company_names.get(
                str(task["Company ID"]), "Unknown"
            ),
            "Contact": contact_names.get(
                str(task["Contact ID"]), ""
            ),
            "Task": task["Task Description"],
            "Priority": task["Priority"],
            "Due Date": due.isoformat(),
            "Days Overdue": max((today - due).days, 0)
        }

        if due < today:
            overdue.append(item)
        elif due == today:
            due_today.append(item)
        elif due <= cutoff:
            upcoming.append(item)

    overdue.sort(
        key=lambda x: (-x["Days Overdue"], x["Due Date"])
    )
    due_today.sort(key=lambda x: x["Priority"])
    upcoming.sort(key=lambda x: x["Due Date"])

    # 2. Proposals requiring decisions
    pending_proposals = []

    for proposal in proposals:
        if proposal["Status"] not in ["Sent", "Under Review"]:
            continue

        proposal_date = parse_date(proposal["Proposal Date"])

        days_waiting = (
            (today - proposal_date).days
            if proposal_date else None
        )

        pending_proposals.append({
            "Proposal ID": proposal["Proposal ID"],
            "Company": company_names.get(
                str(proposal["Company ID"]), "Unknown"
            ),
            "Service": proposal["Service Package"],
            "Total Value": crm_number(proposal["Total Price"]),
            "Status": proposal["Status"],
            "Days Waiting": days_waiting
        })

    pending_proposals.sort(
        key=lambda x: (
            x["Days Waiting"] is None,
            -(x["Days Waiting"] or 0)
        )
    )

    # 3. Highest-priority open opportunities
    priority_opportunities = []

    for opportunity in opportunities:
        if str(opportunity["Outcome"]).strip() in [
            "Won", "Lost"
        ]:
            continue

        priority_opportunities.append({
            "Opportunity ID": opportunity["Opportunity ID"],
            "Company": company_names.get(
                str(opportunity["Company ID"]), "Unknown"
            ),
            "Opportunity": opportunity["Opportunity Name"],
            "Stage": opportunity["Pipeline Stage"],
            "Score": crm_number(
                opportunity["Opportunity Score"]
            ),
            "Estimated Value": crm_number(
                opportunity["Estimated Value"]
            )
        })

    priority_opportunities.sort(
        key=lambda x: (
            -x["Score"],
            -x["Estimated Value"]
        )
    )

    # 4. Today's sales activity
    todays_activities = [
        a for a in activities
        if parse_date(a["Activity Date"]) == today
    ]

    activity_breakdown = {}

    for activity in todays_activities:
        activity_type = str(activity["Activity Type"])
        activity_breakdown[activity_type] = (
            activity_breakdown.get(activity_type, 0) + 1
        )

    # Daily summary
    print("\n" + "=" * 65)
    print("NORTH STAR ANALYTICS")
    print("DAILY SALES COMMAND CENTER")
    print("Date:", today.strftime("%B %d, %Y"))
    print("=" * 65)

    print("Overdue Follow-Ups:", len(overdue))
    print("Due Today:", len(due_today))
    print("Upcoming Follow-Ups:", len(upcoming))
    print("Pending Proposals:", len(pending_proposals))
    print("Open Opportunities:", len(priority_opportunities))
    print("Sales Activities Today:", len(todays_activities))

    task_columns = [
        "Task ID", "Company", "Contact",
        "Task", "Priority", "Due Date", "Days Overdue"
    ]

    show_section(
        "1. OVERDUE FOLLOW-UPS",
        overdue,
        task_columns
    )

    show_section(
        "2. FOLLOW-UPS DUE TODAY",
        due_today,
        task_columns
    )

    show_section(
        f"3. UPCOMING FOLLOW-UPS ({days_ahead} DAYS)",
        upcoming,
        task_columns
    )

    show_section(
        "4. PROPOSALS AWAITING DECISIONS",
        pending_proposals,
        [
            "Proposal ID", "Company", "Service",
            "Total Value", "Status", "Days Waiting"
        ]
    )

    show_section(
        "5. TOP SALES OPPORTUNITIES",
        priority_opportunities,
        [
            "Opportunity ID", "Company", "Opportunity",
            "Stage", "Score", "Estimated Value"
        ],
        limit=top_n
    )

    print("\n" + "=" * 65)
    print("6. TODAY'S OUTREACH ACTIVITY")
    print("=" * 65)

    if activity_breakdown:
        for activity_type, count in sorted(
            activity_breakdown.items()
        ):
            print(f"{activity_type}: {count}")
    else:
        print("No outreach activities recorded today.")

    print("\n✓ Daily command center report complete!")

    return {
        "overdue": overdue,
        "due_today": due_today,
        "upcoming": upcoming,
        "pending_proposals": pending_proposals,
        "priority_opportunities": priority_opportunities,
        "activity_breakdown": activity_breakdown
    }


In [ ]:
# STEP 020 — Core function: show_daily_outreach_report / show_weekly_outreach_report

# CRM functions from original cell 22

def show_daily_outreach_report(report_date=None):
    """Compare daily sales activities with outreach goals."""

    if report_date is None:
        report_date = crm_today()
    elif isinstance(report_date, str):
        report_date = date.fromisoformat(report_date)

    activities = activities_sheet.get_all_records()
    goals = outreach_goals_sheet.get_all_records()

    daily_activities = [
        a for a in activities
        if str(a["Activity Date"]).strip()
        == report_date.isoformat()
        and str(a["Direction"]).strip() == "Outbound"
    ]

    rows = []

    for goal in goals:
        if str(goal["Active"]).strip().lower() != "yes":
            continue

        activity_type = str(goal["Activity Type"]).strip()
        target = int(goal["Daily Goal"] or 0)

        actual = sum(
            str(a["Activity Type"]).strip() == activity_type
            for a in daily_activities
        )

        completion = (
            round(actual / target * 100, 1)
            if target > 0 else 0
        )

        rows.append({
            "Activity Type": activity_type,
            "Daily Goal": target,
            "Completed": actual,
            "Remaining": max(target - actual, 0),
            "Goal Completion %": completion
        })

    print("NORTH STAR ANALYTICS")
    print("DAILY OUTREACH REPORT")
    print("Date:", report_date.isoformat())
    print("-" * 50)

    if rows:
        display(pd.DataFrame(rows))
    else:
        print("No active outreach goals configured.")

    unique_companies = len({
        str(a["Company ID"])
        for a in daily_activities
        if str(a["Company ID"]).strip()
    })

    positive = sum(
        str(a["Outcome"]).strip() in positive_outcomes
        for a in daily_activities
    )

    meetings = sum(
        str(a["Outcome"]).strip() in meeting_outcomes
        for a in daily_activities
    )

    print("Outbound Activities:", len(daily_activities))
    print("Unique Companies Contacted:", unique_companies)
    print("Positive Outcomes:", positive)
    print("Meetings Scheduled:", meetings)

    return rows

def show_weekly_outreach_report(week_start=None):
    """Summarize outreach performance Monday through Sunday."""

    if week_start is None:
        today = crm_today()
        week_start = today - timedelta(days=today.weekday())
    elif isinstance(week_start, str):
        week_start = date.fromisoformat(week_start)

    week_end = week_start + timedelta(days=6)

    activities = activities_sheet.get_all_records()
    goals = outreach_goals_sheet.get_all_records()

    weekly_activities = []

    for activity in activities:
        try:
            activity_date = date.fromisoformat(
                str(activity["Activity Date"]).strip()
            )
        except ValueError:
            continue

        if (
            week_start <= activity_date <= week_end
            and str(activity["Direction"]).strip() == "Outbound"
        ):
            weekly_activities.append(activity)

    rows = []

    for goal in goals:
        if str(goal["Active"]).strip().lower() != "yes":
            continue

        activity_type = str(goal["Activity Type"]).strip()
        target = int(goal["Weekly Goal"] or 0)

        actual = sum(
            str(a["Activity Type"]).strip() == activity_type
            for a in weekly_activities
        )

        rows.append({
            "Activity Type": activity_type,
            "Weekly Goal": target,
            "Completed": actual,
            "Remaining": max(target - actual, 0),
            "Goal Completion %": (
                round(actual / target * 100, 1)
                if target > 0 else 0
            )
        })

    print("NORTH STAR ANALYTICS")
    print("WEEKLY OUTREACH REPORT")
    print(f"{week_start} through {week_end}")
    print("-" * 50)

    if rows:
        display(pd.DataFrame(rows))
    else:
        print("No active outreach goals configured.")

    print("Total Outbound Activities:", len(weekly_activities))

    return rows


In [ ]:
# STEP 021 — Core function: show_discovery_departments

# CRM functions from original cell 27

def show_discovery_departments():
    records = question_library_sheet.get_all_records()

    if not records:
        print("Question library is empty.")
        return

    df = pd.DataFrame(records)

    summary = (
        df.groupby("Department")
        .size()
        .reset_index(name="Questions")
        .sort_values("Department")
    )

    display(summary)
    return summary


In [ ]:
# STEP 022 — Core function: create_discovery_interview / show_discovery_interview

# CRM functions from original cell 28

def create_discovery_interview(
    company_id,
    department,
    contact_id=""
):
    """
    Generate a structured dealership discovery interview
    from the existing question library.
    """

    # Validate company
    companies = companies_sheet.get_all_records()

    company = next(
        (
            c for c in companies
            if str(c["Company ID"]) == str(company_id)
        ),
        None
    )

    if company is None:
        raise ValueError("Company ID not found.")

    # Validate contact belongs to company
    if contact_id:
        contacts = contacts_sheet.get_all_records()

        if not any(
            str(c["Contact ID"]) == str(contact_id)
            and str(c["Company ID"]) == str(company_id)
            for c in contacts
        ):
            raise ValueError(
                "Contact does not belong to this company."
            )

    # Retrieve questions for selected department
    questions = [
        q for q in question_library_sheet.get_all_records()
        if str(q["Department"]).strip().lower()
        == department.strip().lower()
    ]

    if not questions:
        raise ValueError(
            "No questions found for that department."
        )

    # Generate one ID shared by the entire interview
    interview_id = "NSA-I-" + uuid4().hex[:12].upper()
    interview_date = crm_today().isoformat()

    rows = []

    for question in questions:
        rows.append([
            interview_id,
            company_id,
            contact_id,
            interview_date,
            department,
            question["Question ID"],
            question["Business Question"],
            "",  # Client response
            "",  # Identified pain point
            "",  # Business impact
            "",  # Available data
            "",  # Follow-up action
            "Not Started"
        ])

    interview_sheet.append_rows(
        rows,
        value_input_option="RAW"
    )

    print("✓ Discovery interview created!")
    print("Interview ID:", interview_id)
    print("Company:", company["Company Name"])
    print("Department:", department)
    print("Questions:", len(rows))
    print("Status: Not Started")
    print()
    print("Open the 'Discovery Interviews' tab to enter answers.")

    return interview_id

def show_discovery_interview(interview_id):
    """
    Display an existing discovery interview
    and its recorded responses.
    """

    records = interview_sheet.get_all_records()

    matches = [
        r for r in records
        if str(r["Interview ID"]) == str(interview_id)
    ]

    if not matches:
        print("Interview ID not found.")
        return

    df = pd.DataFrame(matches)

    display(df[[
        "Question ID",
        "Business Question",
        "Client Response",
        "Identified Pain Point",
        "Business Impact",
        "Follow-Up Action",
        "Status"
    ]])


In [ ]:
# STEP 023 — Core function: transfer_interview_findings / show_interview_findings

# CRM functions from original cell 29

def transfer_interview_findings(interview_id):
    """
    Transfer documented interview findings into Discovery.
    Avoid duplicate transfers using an interview/question
    reference stored in the Discovery Notes column.
    """

    interviews = interview_sheet.get_all_records()
    discovery_records = discovery_sheet.get_all_records()
    question_records = question_library_sheet.get_all_records()

    question_lookup = {
        str(q["Question ID"]): q
        for q in question_records
    }

    matches = [
        (row_number, record)
        for row_number, record in enumerate(interviews, start=2)
        if str(record["Interview ID"]) == str(interview_id)
    ]

    if not matches:
        raise ValueError("Interview ID not found.")

    transferred = []
    skipped = []

    for row_number, record in matches:

        question_id = str(record["Question ID"])
        client_response = str(record["Client Response"]).strip()
        pain_point = str(record["Identified Pain Point"]).strip()

        # Require both an answer and an identified problem
        if not client_response or not pain_point:
            skipped.append(question_id)
            continue

        reference = f"InterviewRef:{interview_id}:{question_id}"

        # Check whether finding was already transferred
        already_transferred = any(
            reference in str(d["Notes"])
            for d in discovery_records
        )

        if already_transferred:
            skipped.append(question_id)
            continue

        question = question_lookup.get(question_id, {})

        technical_question = question.get(
            "Technical Analysis Question", ""
        )

        required_data = question.get("Required Data", "")
        suggested_kpi = question.get("Suggested KPI", "")
        recommended_service = question.get(
            "Recommended Service", ""
        )

        notes = (
            f"{reference}\n"
            f"Business Question: {record['Business Question']}\n"
            f"Client Response: {client_response}\n"
            f"Technical Analysis: {technical_question}\n"
            f"Required Data: {required_data}\n"
            f"Suggested KPI: {suggested_kpi}\n"
            f"Follow-Up Action: {record['Follow-Up Action']}"
        )

        # Use existing Step 14 discovery function
        discovery_id = create_discovery(
            company_id=record["Company ID"],
            contact_id=record["Contact ID"],
            department=record["Department"],
            current_process=record["Business Question"],
            pain_point=pain_point,
            business_impact=record["Business Impact"],
            available_data=record["Available Data"] or required_data,
            current_kpi=suggested_kpi,
            recommended_service=recommended_service,
            notes=notes
        )

        # Update in-memory records to prevent duplicates
        # within the same execution
        discovery_records.append({"Notes": notes})

        # Mark interview question completed
        interview_sheet.update(
            range_name=f"M{row_number}",
            values=[["Completed"]]
        )

        transferred.append({
            "Question ID": question_id,
            "Discovery ID": discovery_id,
            "Pain Point": pain_point,
            "Recommended Service": recommended_service
        })

    print("\nNORTH STAR ANALYTICS")
    print("DISCOVERY INTERVIEW TRANSFER SUMMARY")
    print("-" * 50)
    print("Interview ID:", interview_id)
    print("Findings Transferred:", len(transferred))
    print("Questions Skipped:", len(skipped))

    if transferred:
        display(pd.DataFrame(transferred))

    print("\n✓ Interview transfer complete!")

    return transferred

def show_interview_findings(interview_id):
    """
    Preview documented findings before transferring them.
    Does not modify CRM records.
    """

    interviews = interview_sheet.get_all_records()

    findings = [
        r for r in interviews
        if str(r["Interview ID"]) == str(interview_id)
        and str(r["Client Response"]).strip()
        and str(r["Identified Pain Point"]).strip()
    ]

    if not findings:
        print("No documented findings available.")
        return []

    df = pd.DataFrame(findings)

    display(df[[
        "Question ID",
        "Business Question",
        "Client Response",
        "Identified Pain Point",
        "Business Impact",
        "Available Data",
        "Follow-Up Action"
    ]])

    return findings


In [ ]:
# STEP 024 — Core function: generate_consulting_recommendations

# CRM functions from original cell 30

def generate_consulting_recommendations(company_id):
    """
    Generate consulting recommendations from documented
    discovery records for a specific company.
    """

    companies = companies_sheet.get_all_records()

    company = next(
        (
            c for c in companies
            if str(c["Company ID"]) == str(company_id)
        ),
        None
    )

    if company is None:
        raise ValueError("Company ID not found.")

    discoveries = [
        d for d in discovery_sheet.get_all_records()
        if str(d["Company ID"]) == str(company_id)
        and str(d["Pain Point"]).strip()
    ]

    if not discoveries:
        print("No documented discovery findings available.")
        return []

    recommendations = {}

    for discovery in discoveries:

        service = str(
            discovery["Recommended Service"]
        ).strip()

        if not service:
            service = "General Process Assessment"

        if service not in recommendations:
            catalog = service_recommendations.get(
                service, {}
            )

            recommendations[service] = {
                "Service": service,
                "Departments": set(),
                "Pain Points": [],
                "Business Impacts": [],
                "KPIs": set(),
                "Available Data": set(),
                "Deliverables": catalog.get(
                    "deliverables",
                    default_deliverables
                ),
                "Continuing Services": catalog.get(
                    "continuing_services", []
                )
            }

        item = recommendations[service]

        item["Departments"].add(
            str(discovery["Department"])
        )

        item["Pain Points"].append(
            str(discovery["Pain Point"])
        )

        if discovery["Business Impact"]:
            item["Business Impacts"].append(
                str(discovery["Business Impact"])
            )

        if discovery["Current KPI"]:
            item["KPIs"].add(
                str(discovery["Current KPI"])
            )

        if discovery["Available Data"]:
            item["Available Data"].add(
                str(discovery["Available Data"])
            )

    results = []

    for item in recommendations.values():
        results.append({
            "Service": item["Service"],
            "Departments": ", ".join(
                sorted(item["Departments"])
            ),
            "Pain Point Count": len(item["Pain Points"]),
            "Pain Points": " | ".join(item["Pain Points"]),
            "Business Impacts": " | ".join(
                item["Business Impacts"]
            ),
            "KPIs": " | ".join(sorted(item["KPIs"])),
            "Available Data": " | ".join(
                sorted(item["Available Data"])
            ),
            "Suggested Deliverables": " | ".join(
                item["Deliverables"]
            ),
            "Continuing Services": " | ".join(
                item["Continuing Services"]
            )
        })

    results.sort(
        key=lambda r: r["Pain Point Count"],
        reverse=True
    )

    print("\nNORTH STAR ANALYTICS")
    print("CONSULTING RECOMMENDATION REPORT")
    print("-" * 55)
    print("Company:", company["Company Name"])
    print("Documented Pain Points:", len(discoveries))
    print("Potential Services:", len(results))

    display(pd.DataFrame(results)[[
        "Service",
        "Departments",
        "Pain Point Count",
        "KPIs",
        "Suggested Deliverables",
        "Continuing Services"
    ]])

    print("\nSCOPE REVIEW CHECKLIST")
    print("-" * 55)

    for flag in scope_protection_flags:
        print("[ ]", flag)

    print("\n✓ Recommendation report generated!")

    return results


In [ ]:
# STEP 025 — Core function: create_proposal_scope_draft / show_proposal_scope_draft

# CRM functions from original cell 31

def create_proposal_scope_draft(company_id):
    """
    Generate an internal proposal scope draft using
    existing documented discovery recommendations.

    Does not create a formal proposal or set prices.
    """

    recommendations = generate_consulting_recommendations(
        company_id
    )

    if not recommendations:
        print("No recommendations available.")
        return None

    services = []
    problems = []
    deliverables = []
    required_data = []
    kpis = []
    continuing = []

    for recommendation in recommendations:

        services.append(recommendation["Service"])

        problems.extend(
            recommendation["Pain Points"].split(" | ")
        )

        deliverables.extend(
            recommendation["Suggested Deliverables"].split(" | ")
        )

        required_data.extend(
            recommendation["Available Data"].split(" | ")
        )

        kpis.extend(
            recommendation["KPIs"].split(" | ")
        )

        continuing.extend(
            recommendation["Continuing Services"].split(" | ")
        )

    # Remove empty entries and duplicates while preserving order
    def unique_items(items):
        return list(dict.fromkeys(
            str(item).strip()
            for item in items
            if str(item).strip()
        ))

    services = unique_items(services)
    problems = unique_items(problems)
    deliverables = unique_items(deliverables)
    required_data = unique_items(required_data)
    kpis = unique_items(kpis)
    continuing = unique_items(continuing)

    exclusions = [
        "Work not expressly included in the approved scope",
        "Additional locations unless specifically included",
        "Additional workflows beyond the contracted limit",
        "Major dashboard development unless specified",
        "Extensive historical data cleanup",
        "Custom software development",
        "On-site travel and related expenses",
        "Additional training sessions",
        "Urgent or out-of-schedule work",
        "Ongoing support unless separately contracted"
    ]

    add_on_review = [
        "Additional locations",
        "More than four workflows",
        "Major dashboard development",
        "Extensive historical data cleanup",
        "Custom software development",
        "On-site travel",
        "Additional training sessions",
        "Urgent work outside the established schedule"
    ]

    scope_id = "NSA-SCOPE-" + uuid4().hex[:10].upper()

    row = [
        scope_id,
        company_id,
        crm_today().isoformat(),
        "Internal Draft",
        json.dumps(services),
        json.dumps(problems),
        json.dumps(deliverables),
        json.dumps(required_data),
        json.dumps(kpis),
        json.dumps(exclusions),
        json.dumps(add_on_review),
        json.dumps(continuing),
        "Requires scope, feasibility, and pricing review."
    ]

    scope_sheet.append_row(
        row,
        value_input_option="RAW"
    )

    print("\n✓ Proposal scope draft created!")
    print("Scope Draft ID:", scope_id)
    print("Services:", len(services))
    print("Proposed Deliverables:", len(deliverables))
    print("Continuing-Service Options:", len(continuing))
    print("Status: Internal Draft")
    print("\nNo prices or formal proposals were created.")

    return scope_id

def show_proposal_scope_draft(scope_id):
    """
    Retrieve and display a saved proposal scope draft.
    """

    records = scope_sheet.get_all_records()

    draft = next(
        (
            record for record in records
            if str(record["Scope Draft ID"]) == str(scope_id)
        ),
        None
    )

    if draft is None:
        print("Scope draft not found.")
        return None

    print("\nNORTH STAR ANALYTICS")
    print("PROPOSAL SCOPE DRAFT")
    print("-" * 50)
    print("Draft ID:", draft["Scope Draft ID"])
    print("Company ID:", draft["Company ID"])
    print("Status:", draft["Draft Status"])

    sections = {
        "Recommended Services": "Recommended Services",
        "Business Problems": "Business Problems",
        "Proposed Deliverables": "Proposed Deliverables",
        "Required Data": "Required Data",
        "KPIs": "KPIs",
        "Scope Exclusions": "Scope Exclusions",
        "Add-On Review": "Add-On Review",
        "Continuing Services": "Continuing Services"
    }

    for heading, column in sections.items():

        print(f"\n{heading.upper()}")

        try:
            items = json.loads(draft[column])
        except (json.JSONDecodeError, TypeError):
            items = [str(draft[column])]

        for item in items:
            print("•", item)

    return draft


In [ ]:
# STEP 026 — Core function: approve_proposal_scope / show_pending_scope_approvals

# CRM functions from original cell 32

def approve_proposal_scope(
    scope_id,
    base_price,
    addon_price=0,
    opportunity_id="",
    approval_notes=""
):
    """
    Approve an internal scope draft and create a proposal record.

    Pricing is entered manually.
    Duplicate proposal creation is prevented.
    """

    drafts = scope_sheet.get_all_records()
    proposals = proposals_sheet.get_all_records()

    draft = next(
        (
            d for d in drafts
            if str(d["Scope Draft ID"]) == str(scope_id)
        ),
        None
    )

    if draft is None:
        raise ValueError("Scope draft not found.")

    if draft["Draft Status"] == "Approved":
        raise ValueError("This scope draft is already approved.")

    # Prevent duplicate proposal creation
    reference = f"ScopeDraftRef:{scope_id}"

    if any(
        reference in str(p["Notes"])
        for p in proposals
    ):
        raise ValueError(
            "A proposal already exists for this scope draft."
        )

    # Validate prices
    try:
        base = Decimal(str(base_price))
        addons = Decimal(str(addon_price))
    except InvalidOperation:
        raise ValueError("Prices must be valid numbers.")

    if not base.is_finite() or not addons.is_finite():
        raise ValueError("Prices must be finite numbers.")

    if base < 0 or addons < 0:
        raise ValueError("Prices cannot be negative.")

    total = base + addons

    # Confirm opportunity belongs to the same company
    if opportunity_id:
        opportunities = opportunities_sheet.get_all_records()

        valid_opportunity = any(
            str(o["Opportunity ID"]) == str(opportunity_id)
            and str(o["Company ID"]) == str(draft["Company ID"])
            for o in opportunities
        )

        if not valid_opportunity:
            raise ValueError(
                "Opportunity does not belong to this company."
            )

    # Retrieve the draft's row number
    draft_cells = scope_sheet.col_values(1)

    draft_row = next(
        (
            index for index, value
            in enumerate(draft_cells, start=1)
            if value == scope_id
        ),
        None
    )

    if draft_row is None:
        raise ValueError("Unable to locate scope draft row.")

    # Create formal proposal
    proposal_id = "NSA-P-" + uuid4().hex[:12].upper()

    notes = (
        f"{reference}\n"
        f"Approved Scope Draft: {scope_id}\n"
        f"Approval Notes: {approval_notes}"
    )

    proposal_row = [
        proposal_id,
        opportunity_id,
        draft["Company ID"],
        crm_today().isoformat(),
        draft["Recommended Services"],
        float(base),
        float(addons),
        float(total),
        draft["Proposed Deliverables"],
        draft["Scope Exclusions"],
        "Draft",
        "",
        "",
        notes
    ]

    proposals_sheet.append_row(
        proposal_row,
        value_input_option="RAW"
    )

    # Mark source scope draft as approved
    scope_sheet.update(
        range_name=f"D{draft_row}",
        values=[["Approved"]]
    )

    print("\n✓ Proposal scope approved!")
    print("✓ Proposal record created!")
    print("Proposal ID:", proposal_id)
    print("Company ID:", draft["Company ID"])
    print("Base Price: $", f"{base:,.2f}", sep="")
    print("Add-On Price: $", f"{addons:,.2f}", sep="")
    print("Total Price: $", f"{total:,.2f}", sep="")
    print("Proposal Status: Draft")
    print("\nThe proposal has NOT been sent to the client.")

    return proposal_id

def show_pending_scope_approvals():
    """
    Show scope drafts awaiting approval.
    """

    drafts = scope_sheet.get_all_records()

    pending = [
        d for d in drafts
        if d["Draft Status"] == "Internal Draft"
    ]

    if not pending:
        print("No scope drafts awaiting approval.")
        return []

    df = pd.DataFrame(pending)

    display(df[[
        "Scope Draft ID",
        "Company ID",
        "Created Date",
        "Draft Status",
        "Recommended Services"
    ]])

    return pending


In [ ]:
# STEP 027 — Core function: proposal_list / generate_proposal_document

# CRM functions from original cell 33

def proposal_list(value):
    """Read JSON lists or ordinary text from CRM cells."""

    if value is None or str(value).strip() == "":
        return []

    try:
        parsed = json.loads(str(value))
        if isinstance(parsed, list):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]
    except (json.JSONDecodeError, TypeError):
        pass

    return [
        line.strip(" •-")
        for line in str(value).splitlines()
        if line.strip()
    ]

def generate_proposal_document(proposal_id):
    """
    Generate a client-facing Word proposal from an
    existing CRM proposal record.

    The proposal is saved to Google Drive.
    No CRM statuses are changed.
    """

    proposals = proposals_sheet.get_all_records()
    companies = companies_sheet.get_all_records()

    proposal = next(
        (
            p for p in proposals
            if str(p["Proposal ID"]) == str(proposal_id)
        ),
        None
    )

    if proposal is None:
        raise ValueError("Proposal ID not found.")

    company = next(
        (
            c for c in companies
            if str(c["Company ID"])
            == str(proposal["Company ID"])
        ),
        None
    )

    if company is None:
        raise ValueError("Associated company not found.")

    # Validate proposal amounts
    try:
        base = Decimal(str(proposal["Base Price"]))
        addons = Decimal(str(proposal["Add-On Price"]))
        total = Decimal(str(proposal["Total Price"]))
    except (InvalidOperation, TypeError):
        raise ValueError("Proposal contains invalid pricing.")

    if not all(
        amount.is_finite() and amount >= 0
        for amount in [base, addons, total]
    ):
        raise ValueError("Proposal contains invalid pricing.")

    if abs((base + addons) - total) > Decimal("0.01"):
        raise ValueError(
            "Proposal total does not match base and add-on prices."
        )

    doc = Document()

    section = doc.sections[0]
    section.top_margin = Inches(0.8)
    section.bottom_margin = Inches(0.8)
    section.left_margin = Inches(0.85)
    section.right_margin = Inches(0.85)

    normal = doc.styles["Normal"]
    normal.font.name = "Aptos"
    normal.font.size = Pt(10)
    normal.paragraph_format.space_after = Pt(7)

    for style_name in ["Heading 1", "Heading 2"]:
        style = doc.styles[style_name]
        style.font.name = "Aptos"
        style.font.color.rgb = NAVY

    # Cover / heading
    heading = doc.add_paragraph()
    heading.style = doc.styles["Title"]

    run = heading.add_run("NORTH STAR ANALYTICS")
    run.bold = True
    run.font.color.rgb = NAVY
    run.font.size = Pt(22)

    subtitle = doc.add_paragraph(
        "BUSINESS OPERATIONS & DATA ANALYTICS CONSULTING"
    )
    subtitle.runs[0].font.color.rgb = COBALT
    subtitle.runs[0].font.size = Pt(10)

    doc.add_paragraph("")

    doc.add_heading("Consulting Proposal", level=1)

    info = doc.add_table(rows=4, cols=2)
    info.style = "Light Shading Accent 1"

    details = [
        ("Prepared For", str(company["Company Name"])),
        ("Proposal ID", str(proposal_id)),
        ("Proposal Date", str(proposal["Proposal Date"])),
        ("Proposal Status", str(proposal["Status"]))
    ]

    for row, (label, value) in zip(info.rows, details):
        row.cells[0].text = label
        row.cells[1].text = value

    doc.add_paragraph("")

    # Services
    doc.add_heading("1. Proposed Services", level=1)

    services = proposal_list(proposal["Service Package"])

    if services:
        for service in services:
            doc.add_paragraph(service, style="List Bullet")
    else:
        doc.add_paragraph("Services pending final confirmation.")

    # Scope
    doc.add_heading("2. Scope of Work", level=1)

    scope_items = proposal_list(proposal["Scope"])

    if scope_items:
        for item in scope_items:
            doc.add_paragraph(item, style="List Bullet")
    else:
        doc.add_paragraph("Scope pending final confirmation.")

    # Exclusions
    doc.add_heading("3. Exclusions & Scope Boundaries", level=1)

    exclusions = proposal_list(proposal["Exclusions"])

    if exclusions:
        for item in exclusions:
            doc.add_paragraph(item, style="List Bullet")
    else:
        doc.add_paragraph(
            "Exclusions must be confirmed before this proposal is issued."
        )

    # Pricing
    doc.add_heading("4. Project Investment", level=1)

    pricing = doc.add_table(rows=3, cols=2)
    pricing.style = "Light Shading Accent 1"

    pricing_rows = [
        ("Base Consulting Services", base),
        ("Additional Services / Add-Ons", addons),
        ("Total Proposed Investment", total)
    ]

    for row, (label, amount) in zip(pricing.rows, pricing_rows):
        row.cells[0].text = label
        row.cells[1].text = f"${amount:,.2f}"

    # Next steps
    doc.add_heading("5. Review & Next Steps", level=1)

    doc.add_paragraph(
        "This proposal is provided for review and discussion. "
        "Final deliverables, project timelines, payment terms, "
        "and responsibilities will be confirmed in a mutually "
        "approved agreement before work begins."
    )

    doc.add_paragraph(
        "Services or deliverables outside the approved scope "
        "require separate written authorization."
    )

    # Footer
    footer = section.footer.paragraphs[0]
    footer.text = (
        f"North Star Analytics | Proposal {proposal_id} "
        "| Confidential"
    )
    footer.style = doc.styles["Normal"]

    # Safe filename
    safe_company = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(company["Company Name"])
    ).strip("_")[:60] or "Client"

    filename = f"{safe_company}_{proposal_id}_Proposal.docx"

    filepath = PROPOSAL_FOLDER / filename

    # Avoid silently overwriting an existing proposal
    if filepath.exists():
        raise FileExistsError(
            f"Document already exists: {filepath}. "
            "Review or rename the existing file before regenerating."
        )

    doc.save(filepath)

    print("\n✓ Proposal document generated!")
    print("Company:", company["Company Name"])
    print("Proposal ID:", proposal_id)
    print("Total Investment:", f"${total:,.2f}")
    print("Saved to:", filepath)
    print("\nReview the document before sharing it with a client.")

    return filepath


In [ ]:
# STEP 028 — Core function: review_proposal / show_proposal_review_status

# CRM functions from original cell 34

def review_proposal(
    proposal_id,
    scope_confirmed=False,
    deliverables_confirmed=False,
    timeline_confirmed=False,
    pricing_confirmed=False,
    payment_terms_confirmed=False,
    exclusions_confirmed=False,
    addons_confirmed=False,
    client_details_confirmed=False,
    reviewer_notes=""
):
    """
    Record proposal quality-control review.
    A proposal is client-ready only when all eight
    review criteria are explicitly confirmed.
    """

    proposals = proposals_sheet.get_all_records()

    proposal = next(
        (
            p for p in proposals
            if str(p["Proposal ID"]) == str(proposal_id)
        ),
        None
    )

    if proposal is None:
        raise ValueError("Proposal ID not found.")

    checks = {
        "Scope": scope_confirmed,
        "Deliverables": deliverables_confirmed,
        "Timeline": timeline_confirmed,
        "Pricing": pricing_confirmed,
        "Payment Terms": payment_terms_confirmed,
        "Exclusions": exclusions_confirmed,
        "Add-Ons": addons_confirmed,
        "Client Details": client_details_confirmed
    }

    # Require actual booleans, not truthy strings
    if not all(type(value) is bool for value in checks.values()):
        raise ValueError(
            "Each review criterion must be True or False."
        )

    all_confirmed = all(checks.values())

    status = (
        "Client Ready"
        if all_confirmed
        else "Needs Revision"
    )

    row_values = [
        proposal_id,
        crm_today().isoformat(),
        *["Yes" if value else "No" for value in checks.values()],
        reviewer_notes,
        status
    ]

    existing_reviews = review_sheet.get_all_records()

    matching_rows = [
        index
        for index, record
        in enumerate(existing_reviews, start=2)
        if str(record["Proposal ID"]) == str(proposal_id)
    ]

    # Keep one current review record per proposal
    if matching_rows:
        review_sheet.update(
            range_name=f"A{matching_rows[0]}:L{matching_rows[0]}",
            values=[row_values],
            value_input_option="RAW"
        )
    else:
        review_sheet.append_row(
            row_values,
            value_input_option="RAW"
        )

    print("\nNORTH STAR ANALYTICS")
    print("PROPOSAL QUALITY CONTROL")
    print("-" * 45)
    print("Proposal:", proposal_id)

    for item, confirmed in checks.items():
        symbol = "✓" if confirmed else "✗"
        print(f"{symbol} {item}")

    print("\nReview Status:", status)

    if all_confirmed:
        print("✓ All eight review criteria confirmed.")
        print("✓ Proposal approved as client-ready.")
    else:
        print("Proposal requires additional review.")

    print("\nCRM proposal status was not changed.")
    print("No document was sent.")

    return status

def show_proposal_review_status(proposal_id):
    """
    Retrieve the latest stored quality-control review.
    """

    records = review_sheet.get_all_records()

    matches = [
        r for r in records
        if str(r["Proposal ID"]) == str(proposal_id)
    ]

    if not matches:
        print("No review recorded for this proposal.")
        return None

    review = matches[-1]

    print("Proposal:", proposal_id)
    print("Review Date:", review["Review Date"])
    print("Review Status:", review["Review Status"])

    return review


In [ ]:
# STEP 029 — Core function: show_proposal_pipeline

# CRM functions from original cell 35

def show_proposal_pipeline():
    """
    Display proposals with company names, review status,
    decision dates, financial values, and follow-up indicators.

    This function does not modify CRM data.
    """

    proposals = proposals_sheet.get_all_records()
    companies = companies_sheet.get_all_records()
    reviews = review_sheet.get_all_records()
    followups = followups_sheet.get_all_records()

    if not proposals:
        print("No proposals recorded yet.")
        return pd.DataFrame()

    company_lookup = {
        str(c["Company ID"]): c["Company Name"]
        for c in companies
    }

    review_lookup = {
        str(r["Proposal ID"]): r["Review Status"]
        for r in reviews
    }

    today = crm_today()
    results = []

    for proposal in proposals:

        proposal_id = str(proposal["Proposal ID"])
        company_id = str(proposal["Company ID"])
        status = str(proposal["Status"]).strip()

        decision_date_text = str(
            proposal["Decision Date"]
        ).strip()

        decision_date = None

        if decision_date_text:
            try:
                decision_date = date.fromisoformat(
                    decision_date_text[:10]
                )
            except ValueError:
                pass

        # Determine decision timing
        if status.lower() in [
            "accepted", "declined", "lost", "won"
        ]:
            timing = "Closed"

        elif decision_date is None:
            timing = "No Decision Date"

        elif decision_date < today:
            timing = "Overdue"

        elif decision_date == today:
            timing = "Due Today"

        else:
            timing = "Upcoming"

        # Find open follow-up tasks associated with
        # the proposal's company.
        # The current Follow-Ups schema does not have
        # a dedicated Proposal ID column.
        related_followups = [
            task for task in followups
            if str(task["Company ID"]) == company_id
            and str(task["Status"]).strip().lower()
            not in ["completed", "cancelled", "canceled"]
        ]

        overdue_followups = 0

        for task in related_followups:
            due_text = str(task["Due Date"]).strip()

            try:
                due_date = date.fromisoformat(due_text[:10])

                if due_date < today:
                    overdue_followups += 1

            except ValueError:
                continue

        try:
            total_value = float(proposal["Total Price"])
        except (ValueError, TypeError):
            total_value = 0.0

        results.append({
            "Proposal ID": proposal_id,
            "Company": company_lookup.get(
                company_id, company_id
            ),
            "Proposal Status": status,
            "Review Status": review_lookup.get(
                proposal_id, "Not Reviewed"
            ),
            "Total Value": total_value,
            "Decision Date": decision_date_text,
            "Decision Timing": timing,
            "Open Company Follow-Ups": len(related_followups),
            "Overdue Company Follow-Ups": overdue_followups
        })

    df = pd.DataFrame(results)

    # Sort urgent decision dates to the top
    timing_order = {
        "Overdue": 0,
        "Due Today": 1,
        "Upcoming": 2,
        "No Decision Date": 3,
        "Closed": 4
    }

    df["_sort"] = df["Decision Timing"].map(timing_order)

    df = df.sort_values(
        ["_sort", "Decision Date", "Company"]
    ).drop(columns="_sort")

    print("\nNORTH STAR ANALYTICS")
    print("PROPOSAL PIPELINE")
    print("-" * 55)

    print("Total Proposals:", len(df))

    open_df = df[
        df["Decision Timing"] != "Closed"
    ]

    print("Open Proposals:", len(open_df))
    print(
        "Open Proposal Value:",
        f"${open_df['Total Value'].sum():,.2f}"
    )

    print(
        "Overdue Decisions:",
        len(df[df["Decision Timing"] == "Overdue"])
    )

    print(
        "Awaiting Quality Review:",
        len(df[
            df["Review Status"] != "Client Ready"
        ])
    )

    display(df)

    return df


In [ ]:
# STEP 030 — Core function: add_business_days / schedule_proposal_followup

# CRM functions from original cell 36

def add_business_days(start_date, business_days):
    """Add weekdays, excluding Saturday and Sunday."""

    if type(business_days) is not int or business_days < 0:
        raise ValueError(
            "business_days must be a nonnegative integer."
        )

    current = start_date
    added = 0

    while added < business_days:
        current += timedelta(days=1)

        if current.weekday() < 5:
            added += 1

    return current

def schedule_proposal_followup(
    proposal_id,
    business_days=3,
    priority="High",
    contact_id=""
):
    """
    Schedule one follow-up for a proposal.

    Prevents duplicate open follow-up tasks
    for the same proposal.
    """

    proposals = proposals_sheet.get_all_records()

    proposal = next(
        (
            p for p in proposals
            if str(p["Proposal ID"]) == str(proposal_id)
        ),
        None
    )

    if proposal is None:
        raise ValueError("Proposal ID not found.")

    proposal_status = str(
        proposal["Status"]
    ).strip().lower()

    if proposal_status in [
        "accepted", "declined", "won", "lost"
    ]:
        raise ValueError(
            "Cannot schedule follow-up for a closed proposal."
        )

    company_id = str(proposal["Company ID"])
    opportunity_id = str(
        proposal["Opportunity ID"]
    )

    # Confirm contact belongs to the proposal company
    if contact_id:
        contacts = contacts_sheet.get_all_records()

        valid_contact = any(
            str(c["Contact ID"]) == str(contact_id)
            and str(c["Company ID"]) == company_id
            for c in contacts
        )

        if not valid_contact:
            raise ValueError(
                "Contact does not belong to proposal company."
            )

    reference = f"ProposalFollowUpRef:{proposal_id}"

    existing_tasks = followups_sheet.get_all_records()

    duplicate = any(
        reference in str(task["Notes"])
        and str(task["Status"]).strip().lower()
        not in ["completed", "cancelled", "canceled"]
        for task in existing_tasks
    )

    if duplicate:
        print(
            "An open follow-up already exists "
            "for this proposal."
        )
        return None

    due_date = add_business_days(
        crm_today(),
        business_days
    )

    task_id = "NSA-T-" + uuid4().hex[:12].upper()

    task_row = [
        task_id,
        company_id,
        contact_id,
        opportunity_id,
        f"Follow up on proposal {proposal_id}",
        "Proposal Follow-Up",
        priority,
        due_date.isoformat(),
        "Open",
        "",
        reference
    ]

    followups_sheet.append_row(
        task_row,
        value_input_option="RAW"
    )

    print("\n✓ Proposal follow-up scheduled!")
    print("Task ID:", task_id)
    print("Proposal ID:", proposal_id)
    print("Due Date:", due_date.isoformat())
    print("Priority:", priority)
    print("Status: Open")
    print("\nNo message was sent to the client.")

    return task_id

def show_proposal_followups(proposal_id=None):
    """
    Display proposal-specific follow-up tasks.
    Optionally filter by proposal ID.
    """

    tasks = followups_sheet.get_all_records()

    matches = []

    for task in tasks:
        notes = str(task["Notes"])

        if "ProposalFollowUpRef:" not in notes:
            continue

        if proposal_id:
            expected = f"ProposalFollowUpRef:{proposal_id}"

            if expected not in notes:
                continue

        matches.append(task)

    if not matches:
        print("No proposal follow-up tasks found.")
        return []

    df = pd.DataFrame(matches)

    display(df[[
        "Task ID",
        "Company ID",
        "Task Description",
        "Priority",
        "Due Date",
        "Status"
    ]])

    return matches


In [ ]:
# STEP 031 — Core function: search_communication_templates

# CRM functions from original cell 37

def search_communication_templates(
    pipeline_stage=None,
    keyword=None
):
    """Search the communication template library."""

    records = template_sheet.get_all_records()

    df = pd.DataFrame(records)

    if df.empty:
        print("No communication templates available.")
        return df

    if pipeline_stage:
        df = df[
            df["Pipeline Stage"].str.contains(
                pipeline_stage,
                case=False,
                na=False,
                regex=False
            )
        ]

    if keyword:
        mask = (
            df["Template Name"].str.contains(
                keyword, case=False, na=False, regex=False
            )
            |
            df["Message"].str.contains(
                keyword, case=False, na=False, regex=False
            )
        )

        df = df[mask]

    display(df[[
        "Template ID",
        "Template Name",
        "Pipeline Stage",
        "Communication Type",
        "Recommended Timing"
    ]])

    return df


In [ ]:
# STEP 032 — Core function: generate_client_message

# CRM functions from original cell 39

def generate_client_message(
    template_id,
    company_id,
    contact_id,
    meeting_details="",
    requested_data="",
    next_steps="",
    additional_fields=None
):
    """
    Personalize an existing communication template
    using company and contact information.

    Returns a draft. Does not send messages or
    modify CRM records.
    """

    templates = template_sheet.get_all_records()
    companies = companies_sheet.get_all_records()
    contacts = contacts_sheet.get_all_records()

    # Find template
    template = next(
        (
            t for t in templates
            if str(t["Template ID"]) == str(template_id)
        ),
        None
    )

    if template is None:
        raise ValueError("Template ID not found.")

    if str(template["Active"]).strip().lower() not in [
        "yes", "true", "active"
    ]:
        raise ValueError("This communication template is inactive.")

    # Find company
    company = next(
        (
            c for c in companies
            if str(c["Company ID"]) == str(company_id)
        ),
        None
    )

    if company is None:
        raise ValueError("Company ID not found.")

    # Find contact
    contact = next(
        (
            c for c in contacts
            if str(c["Contact ID"]) == str(contact_id)
            and str(c["Company ID"]) == str(company_id)
        ),
        None
    )

    if contact is None:
        raise ValueError(
            "Contact ID not found for the selected company."
        )

    fields = {
        "first_name": str(contact["First Name"]).strip(),
        "last_name": str(contact["Last Name"]).strip(),
        "company_name": str(company["Company Name"]).strip(),
        "contact_title": str(contact["Job Title"]).strip(),
        "meeting_details": str(meeting_details).strip(),
        "requested_data": str(requested_data).strip(),
        "next_steps": str(next_steps).strip()
    }

    # Allow additional custom placeholder values
    if additional_fields is not None:
        if not isinstance(additional_fields, dict):
            raise ValueError("additional_fields must be a dictionary.")

        for key, value in additional_fields.items():
            if key not in fields:
                fields[key] = str(value).strip()

    subject_template = str(template["Subject"])
    message_template = str(template["Message"])

    # Identify required placeholders
    formatter = Formatter()

    required_fields = set()

    for text_value in [subject_template, message_template]:
        for _, field_name, _, _ in formatter.parse(text_value):
            if field_name:
                required_fields.add(field_name)

    missing = [
        field_name
        for field_name in sorted(required_fields)
        if field_name not in fields
        or not fields[field_name]
    ]

    if missing:
        raise ValueError(
            "Missing information for template: "
            + ", ".join(missing)
        )

    # Fill placeholders
    subject = subject_template.format_map(fields)
    message = message_template.format_map(fields)

    result = {
        "Template ID": template_id,
        "Company ID": company_id,
        "Contact ID": contact_id,
        "Recipient Name": (
            fields["first_name"] + " " + fields["last_name"]
        ).strip(),
        "Recipient Email": str(contact["Email"]).strip(),
        "Subject": subject,
        "Message": message,
        "Recommended Timing": template["Recommended Timing"]
    }

    print("\nNORTH STAR ANALYTICS")
    print("PERSONALIZED CLIENT MESSAGE")
    print("-" * 55)
    print("To:", result["Recipient Name"])
    print("Email:", result["Recipient Email"])
    print("Subject:", subject)
    print("-" * 55)
    print(message)
    print("-" * 55)
    print("Recommended Timing:", result["Recommended Timing"])
    print("\n✓ Personalized draft generated!")
    print("No message was sent or logged.")

    return result


In [ ]:
# STEP 033 — Core function: record_client_communication / show_contact_communication_history

# CRM functions from original cell 40

def record_client_communication(
    message,
    activity_type="Email",
    outcome="Sent",
    notes="",
    followup_days=None,
    followup_priority="Medium"
):
    """
    Record a communication that actually occurred.

    Optionally schedule a follow-up task.

    The message must be a dictionary returned by
    generate_client_message().

    No email or message is sent by this function.
    """

    if not isinstance(message, dict):
        raise ValueError(
            "message must be generated by "
            "generate_client_message()."
        )

    required_keys = [
        "Template ID",
        "Company ID",
        "Contact ID",
        "Subject",
        "Message"
    ]

    missing = [
        key for key in required_keys
        if key not in message
    ]

    if missing:
        raise ValueError(
            "Missing message fields: " + ", ".join(missing)
        )

    # Verify company/contact relationship
    company_id = str(message["Company ID"])
    contact_id = str(message["Contact ID"])

    valid_contact = any(
        str(c["Contact ID"]) == contact_id
        and str(c["Company ID"]) == company_id
        for c in contacts_sheet.get_all_records()
    )

    if not valid_contact:
        raise ValueError(
            "Contact is not associated with this company."
        )

    if followup_days is not None:
        if type(followup_days) is not int or followup_days < 0:
            raise ValueError(
                "followup_days must be a nonnegative integer."
            )

    today = crm_today()

    # Create an activity record
    activity_id = "NSA-A-" + uuid4().hex[:12].upper()

    activity_notes = (
        f"Template: {message['Template ID']}\n"
        f"Subject: {message['Subject']}\n"
        f"Outcome: {outcome}\n"
        f"Notes: {notes}"
    )

    activity_row = [
        activity_id,
        company_id,
        contact_id,
        today.isoformat(),
        activity_type,
        "Outbound",
        outcome,
        activity_notes,
        "",
        ""
    ]

    # Prepare optional follow-up
    task_id = None
    task_row = None

    if followup_days is not None:

        due_date = today + timedelta(days=followup_days)
        task_id = "NSA-T-" + uuid4().hex[:12].upper()

        task_row = [
            task_id,
            company_id,
            contact_id,
            "",
            f"Follow up: {message['Subject']}",
            "Communication Follow-Up",
            followup_priority,
            due_date.isoformat(),
            "Open",
            "",
            f"RelatedActivity:{activity_id}"
        ]

        # Store follow-up details on the activity
        activity_row[8] = task_row[4]
        activity_row[9] = due_date.isoformat()

    # Save activity first
    activities_sheet.append_row(
        activity_row,
        value_input_option="RAW"
    )

    # Save optional follow-up task
    if task_row is not None:
        followups_sheet.append_row(
            task_row,
            value_input_option="RAW"
        )

    print("\n✓ Client communication recorded!")
    print("Activity ID:", activity_id)
    print("Company ID:", company_id)
    print("Contact ID:", contact_id)
    print("Activity Type:", activity_type)
    print("Outcome:", outcome)

    if task_id:
        print("\n✓ Follow-up task created!")
        print("Task ID:", task_id)
        print("Due Date:", due_date.isoformat())

    print("\nNo message was sent automatically.")

    return {
        "Activity ID": activity_id,
        "Follow-Up Task ID": task_id
    }

def show_contact_communication_history(contact_id):
    """
    Display recorded communication activities
    for a specific contact.
    """

    activities = activities_sheet.get_all_records()

    matches = [
        a for a in activities
        if str(a["Contact ID"]) == str(contact_id)
    ]

    if not matches:
        print("No recorded communication history.")
        return []

    df = pd.DataFrame(matches)

    df = df.sort_values(
        "Activity Date",
        ascending=False
    )

    display(df[[
        "Activity ID",
        "Activity Date",
        "Activity Type",
        "Direction",
        "Outcome",
        "Notes",
        "Follow-Up Date"
    ]])

    return matches


In [ ]:
# STEP 034 — Core function: north_star_daily_command_center

# CRM functions from original cell 41

def north_star_daily_command_center():
    """
    Display daily CRM priorities and outreach progress.

    Read-only: does not create or modify records.
    """

    today = crm_today()
    today_text = today.isoformat()

    companies = companies_sheet.get_all_records()
    contacts = contacts_sheet.get_all_records()
    activities = activities_sheet.get_all_records()
    tasks = followups_sheet.get_all_records()
    opportunities = opportunities_sheet.get_all_records()
    goals = outreach_goals_sheet.get_all_records()

    company_lookup = {
        str(c["Company ID"]): str(c["Company Name"])
        for c in companies
    }

    print("=" * 65)
    print("NORTH STAR ANALYTICS")
    print("DAILY OUTREACH COMMAND CENTER")
    print("Date:", today_text)
    print("=" * 65)

    # -------------------------------------------------
    # 1. CRM OVERVIEW
    # -------------------------------------------------

    open_tasks = [
        task for task in tasks
        if str(task["Status"]).strip().lower()
        not in ["completed", "cancelled", "canceled"]
    ]

    open_opportunities = [
        opportunity for opportunity in opportunities
        if str(opportunity["Pipeline Stage"]).strip().lower()
        not in ["won", "lost"]
        and str(opportunity["Outcome"]).strip().lower()
        not in ["won", "lost"]
    ]

    print("\nCRM OVERVIEW")
    print("-" * 65)
    print("Companies:", len(companies))
    print("Contacts:", len(contacts))
    print("Open Opportunities:", len(open_opportunities))
    print("Open Tasks:", len(open_tasks))

    # -------------------------------------------------
    # 2. TODAY'S OUTREACH GOALS
    # -------------------------------------------------

    today_activities = [
        activity for activity in activities
        if str(activity["Activity Date"])[:10] == today_text
        and str(activity["Direction"]).strip().lower()
        == "outbound"
    ]

    print("\nTODAY'S OUTREACH GOALS")
    print("-" * 65)

    outreach_rows = []

    for goal in goals:

        if str(goal["Active"]).strip().lower() not in [
            "yes", "true", "active"
        ]:
            continue

        activity_type = str(goal["Activity Type"]).strip()

        try:
            daily_goal = int(float(goal["Daily Goal"]))
        except (ValueError, TypeError):
            daily_goal = 0

        completed = sum(
            1 for activity in today_activities
            if str(activity["Activity Type"]).strip().lower()
            == activity_type.lower()
            and str(activity["Outcome"]).strip().lower()
            not in ["draft", "planned", "not sent", "cancelled"]
        )

        outreach_rows.append({
            "Activity Type": activity_type,
            "Daily Goal": daily_goal,
            "Recorded Activity": completed,
            "Remaining": max(0, daily_goal - completed),
            "Progress %": (
                round(completed / daily_goal * 100, 1)
                if daily_goal > 0 else 0
            )
        })

    if outreach_rows:
        display(pd.DataFrame(outreach_rows))
    else:
        print("No active outreach goals configured.")

    # -------------------------------------------------
    # 3. OVERDUE AND TODAY'S FOLLOW-UPS
    # -------------------------------------------------

    urgent_tasks = []

    for task in open_tasks:

        due_text = str(task["Due Date"]).strip()

        if not due_text:
            continue

        try:
            due_date = date.fromisoformat(due_text[:10])
        except ValueError:
            continue

        if due_date <= today:

            urgent_tasks.append({
                "Task ID": task["Task ID"],
                "Company": company_lookup.get(
                    str(task["Company ID"]),
                    str(task["Company ID"])
                ),
                "Task": task["Task Description"],
                "Priority": task["Priority"],
                "Due Date": due_text,
                "Timing": (
                    "Overdue" if due_date < today else "Due Today"
                )
            })

    print("\nFOLLOW-UPS REQUIRING ATTENTION")
    print("-" * 65)

    if urgent_tasks:
        urgent_df = pd.DataFrame(urgent_tasks)
        urgent_df = urgent_df.sort_values("Due Date")
        display(urgent_df)
    else:
        print("No overdue or due-today follow-ups.")

    # -------------------------------------------------
    # 4. UPCOMING FOLLOW-UPS (NEXT 7 DAYS)
    # -------------------------------------------------

    upcoming_tasks = []

    for task in open_tasks:

        due_text = str(task["Due Date"]).strip()

        try:
            due_date = date.fromisoformat(due_text[:10])
        except ValueError:
            continue

        if today < due_date <= today + timedelta(days=7):

            upcoming_tasks.append({
                "Company": company_lookup.get(
                    str(task["Company ID"]),
                    str(task["Company ID"])
                ),
                "Task": task["Task Description"],
                "Due Date": due_text,
                "Priority": task["Priority"]
            })

    print("\nUPCOMING FOLLOW-UPS — NEXT 7 DAYS")
    print("-" * 65)

    if upcoming_tasks:
        display(
            pd.DataFrame(upcoming_tasks).sort_values("Due Date")
        )
    else:
        print("No upcoming follow-ups in the next 7 days.")

    # -------------------------------------------------
    # 5. PRIORITY OPPORTUNITIES
    # -------------------------------------------------

    print("\nPRIORITY SALES OPPORTUNITIES")
    print("-" * 65)

    priority_rows = []

    for opportunity in open_opportunities:

        try:
            score = float(
                opportunity["Opportunity Score"]
            )
        except (ValueError, TypeError):
            score = 0

        priority_rows.append({
            "Company": company_lookup.get(
                str(opportunity["Company ID"]),
                str(opportunity["Company ID"])
            ),
            "Opportunity": opportunity["Opportunity Name"],
            "Pipeline Stage": opportunity["Pipeline Stage"],
            "Score": score,
            "Estimated Value": opportunity["Estimated Value"]
        })

    if priority_rows:
        priority_df = pd.DataFrame(priority_rows)
        priority_df = priority_df.sort_values(
            "Score", ascending=False
        )
        display(priority_df.head(10))
    else:
        print("No open sales opportunities.")

    # -------------------------------------------------
    # 6. RECENT COMMUNICATIONS
    # -------------------------------------------------

    print("\nRECENT OUTREACH ACTIVITY")
    print("-" * 65)

    recent = sorted(
        [
            a for a in activities
            if str(a["Direction"]).strip().lower()
            == "outbound"
        ],
        key=lambda a: str(a["Activity Date"]),
        reverse=True
    )[:10]

    if recent:
        recent_rows = []

        for activity in recent:
            recent_rows.append({
                "Date": activity["Activity Date"],
                "Company": company_lookup.get(
                    str(activity["Company ID"]),
                    str(activity["Company ID"])
                ),
                "Type": activity["Activity Type"],
                "Outcome": activity["Outcome"],
                "Next Action": activity["Next Action"]
            })

        display(pd.DataFrame(recent_rows))
    else:
        print("No outreach activity recorded.")

    print("\n" + "=" * 65)
    print("END OF DAILY COMMAND CENTER")
    print("=" * 65)

    return {
        "Date": today_text,
        "Companies": len(companies),
        "Contacts": len(contacts),
        "Open Opportunities": len(open_opportunities),
        "Open Tasks": len(open_tasks),
        "Due or Overdue Tasks": len(urgent_tasks),
        "Upcoming Tasks": len(upcoming_tasks),
        "Outreach Activity Today": len(today_activities)
    }


In [ ]:
# STEP 035 — Core function: export_daily_crm_report

# CRM functions from original cell 42

def export_daily_crm_report():
    """
    Export a dated CRM report to Google Drive.

    Creates a multi-sheet Excel workbook.
    Does not modify CRM data.
    """

    today = crm_today()
    today_text = today.isoformat()

    companies = companies_sheet.get_all_records()
    contacts = contacts_sheet.get_all_records()
    activities = activities_sheet.get_all_records()
    tasks = followups_sheet.get_all_records()
    opportunities = opportunities_sheet.get_all_records()
    goals = outreach_goals_sheet.get_all_records()

    company_lookup = {
        str(c["Company ID"]): str(c["Company Name"])
        for c in companies
    }

    # -----------------------------------------
    # Outreach progress
    # -----------------------------------------

    today_outreach = [
        a for a in activities
        if str(a["Activity Date"])[:10] == today_text
        and str(a["Direction"]).strip().lower() == "outbound"
    ]

    outreach_rows = []

    for goal in goals:

        if str(goal["Active"]).strip().lower() not in [
            "yes", "true", "active"
        ]:
            continue

        activity_type = str(goal["Activity Type"]).strip()

        try:
            daily_goal = int(float(goal["Daily Goal"]))
        except (ValueError, TypeError):
            daily_goal = 0

        completed = sum(
            1 for a in today_outreach
            if str(a["Activity Type"]).strip().lower()
            == activity_type.lower()
            and str(a["Outcome"]).strip().lower()
            not in ["draft", "planned", "not sent", "cancelled"]
        )

        outreach_rows.append({
            "Activity Type": activity_type,
            "Daily Goal": daily_goal,
            "Completed": completed,
            "Remaining": max(0, daily_goal - completed)
        })

    # -----------------------------------------
    # Follow-up tasks
    # -----------------------------------------

    overdue_rows = []
    upcoming_rows = []

    for task in tasks:

        status = str(task["Status"]).strip().lower()

        if status in ["completed", "cancelled", "canceled"]:
            continue

        due_text = str(task["Due Date"]).strip()

        try:
            due_date = date.fromisoformat(due_text[:10])
        except ValueError:
            continue

        row = {
            "Task ID": task["Task ID"],
            "Company": company_lookup.get(
                str(task["Company ID"]),
                str(task["Company ID"])
            ),
            "Task": task["Task Description"],
            "Priority": task["Priority"],
            "Due Date": due_text,
            "Status": task["Status"]
        }

        if due_date <= today:
            overdue_rows.append(row)

        elif due_date <= today + timedelta(days=7):
            upcoming_rows.append(row)

    # -----------------------------------------
    # Priority opportunities
    # -----------------------------------------

    opportunity_rows = []

    for opportunity in opportunities:

        if str(opportunity["Pipeline Stage"]).strip().lower() in [
            "won", "lost"
        ]:
            continue

        if str(opportunity["Outcome"]).strip().lower() in [
            "won", "lost"
        ]:
            continue

        try:
            score = float(opportunity["Opportunity Score"])
        except (ValueError, TypeError):
            score = 0

        opportunity_rows.append({
            "Opportunity ID": opportunity["Opportunity ID"],
            "Company": company_lookup.get(
                str(opportunity["Company ID"]),
                str(opportunity["Company ID"])
            ),
            "Opportunity": opportunity["Opportunity Name"],
            "Pipeline Stage": opportunity["Pipeline Stage"],
            "Score": score,
            "Estimated Value": opportunity["Estimated Value"]
        })

    opportunity_rows.sort(
        key=lambda x: x["Score"],
        reverse=True
    )

    # -----------------------------------------
    # Recent communications
    # -----------------------------------------

    recent_rows = []

    recent_activities = sorted(
        activities,
        key=lambda a: str(a["Activity Date"]),
        reverse=True
    )[:20]

    for activity in recent_activities:

        recent_rows.append({
            "Activity ID": activity["Activity ID"],
            "Date": activity["Activity Date"],
            "Company": company_lookup.get(
                str(activity["Company ID"]),
                str(activity["Company ID"])
            ),
            "Activity Type": activity["Activity Type"],
            "Direction": activity["Direction"],
            "Outcome": activity["Outcome"],
            "Next Action": activity["Next Action"]
        })

    # -----------------------------------------
    # Executive summary
    # -----------------------------------------

    summary_rows = [
        ["Report Date", today_text],
        ["Total Companies", len(companies)],
        ["Total Contacts", len(contacts)],
        ["Open Follow-Up Tasks",
         sum(
             str(t["Status"]).strip().lower()
             not in ["completed", "cancelled", "canceled"]
             for t in tasks
         )],
        ["Due or Overdue Follow-Ups", len(overdue_rows)],
        ["Upcoming Follow-Ups", len(upcoming_rows)],
        ["Open Opportunities", len(opportunity_rows)],
        ["Recorded Outbound Activities Today",
         len(today_outreach)]
    ]

    filename = (
        f"North_Star_Analytics_Daily_CRM_"
        f"{today_text}.xlsx"
    )

    filepath = REPORT_FOLDER / filename

    # Write all sections into one workbook
    with pd.ExcelWriter(
        filepath,
        engine="openpyxl"
    ) as writer:

        pd.DataFrame(
            summary_rows,
            columns=["Metric", "Value"]
        ).to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        pd.DataFrame(outreach_rows).to_excel(
            writer,
            sheet_name="Outreach Goals",
            index=False
        )

        pd.DataFrame(overdue_rows).to_excel(
            writer,
            sheet_name="Due Follow-Ups",
            index=False
        )

        pd.DataFrame(upcoming_rows).to_excel(
            writer,
            sheet_name="Upcoming Follow-Ups",
            index=False
        )

        pd.DataFrame(opportunity_rows).to_excel(
            writer,
            sheet_name="Opportunities",
            index=False
        )

        pd.DataFrame(recent_rows).to_excel(
            writer,
            sheet_name="Recent Activities",
            index=False
        )

    print("\n✓ Daily CRM report exported!")
    print("Report Date:", today_text)
    print("Saved to:", filepath)
    print("Worksheets: 6")
    print("\nYour CRM data was not modified.")

    return filepath


In [ ]:
# STEP 036 — Company entry form

# NORTH STAR ANALYTICS
# Prospect Management Center — Add Company Form


company_name = widgets.Text(
    description="Company:",
    placeholder="Enter business name",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="550px")
)

industry = widgets.Dropdown(
    options=[
        "Automotive", "Retail", "Professional Services",
        "Manufacturing", "Healthcare", "Hospitality",
        "Nonprofit", "Other"
    ],
    description="Industry:",
    style={"description_width": "130px"}
)

business_type = widgets.Text(
    value="Automotive Dealership",
    description="Business Type:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="550px")
)

website = widgets.Text(description="Website:")
phone = widgets.Text(description="Phone:")
email = widgets.Text(description="Email:")
address = widgets.Text(description="Address:")
city = widgets.Text(description="City:")
state = widgets.Text(description="State:")

number_of_locations = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10000,
    description="Locations:"
)

company_size = widgets.Text(
    description="Company Size:",
    placeholder="e.g., 25 employees"
)

lead_source = widgets.Dropdown(
    options=[
        "Cold Outreach",
        "Referral",
        "LinkedIn",
        "Website",
        "Networking",
        "Existing Relationship",
        "Other"
    ],
    description="Lead Source:"
)

priority = widgets.Dropdown(
    options=["High", "Medium", "Low"],
    value="Medium",
    description="Priority:"
)

tags = widgets.Text(
    description="Tags:",
    placeholder="Comma-separated tags"
)

notes = widgets.Textarea(
    description="Notes:",
    placeholder="Research findings, business needs, etc.",
    layout=widgets.Layout(width="550px", height="100px")
)

save_button = widgets.Button(
    description="Save Company",
    button_style="success",
    icon="save"
)

company_output = widgets.Output()

def save_new_company(_):
    with company_output:
        clear_output()

        if not company_name.value.strip():
            print("⚠ Company name is required.")
            return

        save_button.disabled = True

        try:
            result = create_company(
                company_name=company_name.value.strip(),
                industry=industry.value,
                business_type=business_type.value.strip(),
                website=website.value.strip(),
                phone=phone.value.strip(),
                email=email.value.strip(),
                address=address.value.strip(),
                city=city.value.strip(),
                state=state.value.strip(),
                number_of_locations=number_of_locations.value,
                company_size=company_size.value.strip(),
                lead_source=lead_source.value,
                priority=priority.value,
                tags=[
                    tag.strip()
                    for tag in tags.value.split(",")
                    if tag.strip()
                ],
                notes=notes.value.strip()
            )

            print("✓ Company saved successfully.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print(
                "Check the Companies worksheet before "
                "trying again to avoid duplicates."
            )

        finally:
            save_button.disabled = False

save_button.on_click(save_new_company)

company_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Add Company</h2>"),
    company_name,
    industry,
    business_type,
    website,
    phone,
    email,
    address,
    city,
    state,
    number_of_locations,
    company_size,
    lead_source,
    priority,
    tags,
    notes,
    save_button,
    company_output
])


In [ ]:
# STEP 037 — Contact entry form

# NORTH STAR ANALYTICS — ADD CONTACT FORM


def get_company_options():
    records = companies_sheet.get_all_records()
    options = [("Select a company...", "")]

    for record in records:
        company_id = str(record.get("Company ID", "")).strip()
        company_name = str(record.get("Company Name", "")).strip()

        if company_id and company_name:
            options.append(
                (f"{company_name} ({company_id})", company_id)
            )

    return options

contact_company = widgets.Dropdown(
    options=get_company_options(),
    description="Company:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px")
)

contact_first_name = widgets.Text(
    description="First Name:",
    style={"description_width": "130px"}
)

contact_last_name = widgets.Text(
    description="Last Name:",
    style={"description_width": "130px"}
)

contact_job_title = widgets.Text(
    description="Job Title:",
    style={"description_width": "130px"}
)

contact_department = widgets.Dropdown(
    options=[
        "Executive Leadership",
        "Sales",
        "Business Development / BDC",
        "Finance and Insurance",
        "Service",
        "Parts",
        "Inventory",
        "Accounting",
        "Human Resources",
        "Marketing",
        "Operations",
        "Customer Experience",
        "Other"
    ],
    description="Department:",
    style={"description_width": "130px"}
)

contact_email = widgets.Text(
    description="Email:",
    style={"description_width": "130px"}
)

contact_phone = widgets.Text(
    description="Phone:",
    style={"description_width": "130px"}
)

contact_decision_maker = widgets.Dropdown(
    options=["Yes", "No", "Influencer", "Unknown"],
    value="Unknown",
    description="Decision-Maker:",
    style={"description_width": "130px"}
)

contact_method = widgets.Dropdown(
    options=["Email", "Phone", "LinkedIn", "In Person"],
    description="Preferred Contact:",
    style={"description_width": "130px"}
)

contact_tags = widgets.Text(
    description="Tags:",
    placeholder="Comma-separated tags",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px")
)

contact_notes = widgets.Textarea(
    description="Notes:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px", height="100px")
)

refresh_companies_button = widgets.Button(
    description="Refresh Companies",
    icon="refresh"
)

save_contact_button = widgets.Button(
    description="Save Contact",
    button_style="success",
    icon="save"
)

contact_output = widgets.Output()

def refresh_contact_companies(_):
    with contact_output:
        clear_output()
        try:
            contact_company.options = get_company_options()
            print("✓ Company list refreshed.")
        except Exception as error:
            print("Unable to refresh companies:", error)

def save_new_contact(_):
    with contact_output:
        clear_output()

        if not contact_company.value:
            print("⚠ Select a company.")
            return

        if not contact_first_name.value.strip():
            print("⚠ First name is required.")
            return

        if not contact_last_name.value.strip():
            print("⚠ Last name is required.")
            return

        save_contact_button.disabled = True

        try:
            result = create_contact(
                company_id=contact_company.value,
                first_name=contact_first_name.value.strip(),
                last_name=contact_last_name.value.strip(),
                job_title=contact_job_title.value.strip(),
                department=contact_department.value,
                email=contact_email.value.strip(),
                phone=contact_phone.value.strip(),
                decision_maker=contact_decision_maker.value,
                preferred_contact_method=contact_method.value,
                tags=[
                    tag.strip()
                    for tag in contact_tags.value.split(",")
                    if tag.strip()
                ],
                notes=contact_notes.value.strip()
            )

            print("✓ Contact saved successfully.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print(
                "Check the Contacts worksheet before retrying "
                "to avoid duplicate records."
            )

        finally:
            save_contact_button.disabled = False

refresh_companies_button.on_click(refresh_contact_companies)
save_contact_button.on_click(save_new_contact)

contact_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Add Contact</h2>"),
    contact_company,
    refresh_companies_button,
    contact_first_name,
    contact_last_name,
    contact_job_title,
    contact_department,
    contact_email,
    contact_phone,
    contact_decision_maker,
    contact_method,
    contact_tags,
    contact_notes,
    save_contact_button,
    contact_output
])


In [ ]:
# STEP 038 — Outreach activity form

# NORTH STAR ANALYTICS — LOG OUTREACH ACTIVITY FORM


def outreach_company_options():
    records = companies_sheet.get_all_records()
    return [("Select a company...", "")] + [
        (
            f"{r.get('Company Name', '')} ({r.get('Company ID', '')})",
            str(r.get("Company ID", "")).strip()
        )
        for r in records
        if r.get("Company ID")
    ]

outreach_company = widgets.Dropdown(
    options=outreach_company_options(),
    description="Company:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px")
)

outreach_contact = widgets.Dropdown(
    options=[("No contact selected", "")],
    description="Contact:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px")
)

def update_outreach_contacts(change=None):
    company_id = outreach_company.value
    options = [("No contact selected", "")]

    if company_id:
        for record in contacts_sheet.get_all_records():
            if str(record.get("Company ID", "")).strip() == company_id:
                name = (
                    str(record.get("First Name", "")).strip()
                    + " "
                    + str(record.get("Last Name", "")).strip()
                ).strip()

                options.append(
                    (name, str(record.get("Contact ID", "")).strip())
                )

    outreach_contact.options = options

outreach_company.observe(
    update_outreach_contacts,
    names="value"
)

outreach_activity_type = widgets.Dropdown(
    options=[
        "Phone",
        "Email",
        "LinkedIn",
        "In-Person",
        "Video",
        "Referral",
        "Other"
    ],
    description="Activity Type:",
    style={"description_width": "130px"}
)

outreach_direction = widgets.Dropdown(
    options=["Outbound", "Inbound"],
    description="Direction:",
    style={"description_width": "130px"}
)

outreach_outcome = widgets.Dropdown(
    options=[
        "No Response",
        "Connected",
        "Positive Response",
        "Interested",
        "Meeting Scheduled",
        "Discovery Scheduled",
        "Information Sent",
        "Follow-Up Needed",
        "Not Interested",
        "Other"
    ],
    description="Outcome:",
    style={"description_width": "130px"}
)

outreach_date = widgets.DatePicker(
    description="Activity Date:",
    value=date.today(),
    style={"description_width": "130px"}
)

outreach_notes = widgets.Textarea(
    description="Notes:",
    placeholder="What happened during this interaction?",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px", height="100px")
)

outreach_next_action = widgets.Text(
    description="Next Action:",
    placeholder="Call again, send proposal, schedule meeting...",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="600px")
)

outreach_followup_date = widgets.DatePicker(
    description="Follow-Up Date:",
    style={"description_width": "130px"}
)

outreach_refresh_button = widgets.Button(
    description="Refresh Companies",
    icon="refresh"
)

outreach_save_button = widgets.Button(
    description="Save Outreach Activity",
    button_style="success",
    icon="save"
)

outreach_output = widgets.Output()

def refresh_outreach_companies(_):
    with outreach_output:
        clear_output()
        outreach_company.options = outreach_company_options()
        update_outreach_contacts()
        print("✓ Company and contact lists refreshed.")

def save_outreach_activity(_):
    with outreach_output:
        clear_output()

        if not outreach_company.value:
            print("⚠ Select a company.")
            return

        outreach_save_button.disabled = True

        try:
            result = log_activity(
                company_id=outreach_company.value,
                activity_type=outreach_activity_type.value,
                outcome=outreach_outcome.value,
                contact_id=outreach_contact.value or "",
                direction=outreach_direction.value,
                notes=outreach_notes.value.strip(),
                next_action=outreach_next_action.value.strip(),
                follow_up_date=(
                    outreach_followup_date.value.isoformat()
                    if outreach_followup_date.value else ""
                ),
                activity_date=(
                    outreach_date.value.isoformat()
                    if outreach_date.value else None
                )
            )

            print("✓ Outreach activity saved.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print(
                "Check the Activities worksheet before "
                "retrying to avoid duplicate entries."
            )

        finally:
            outreach_save_button.disabled = False

outreach_refresh_button.on_click(refresh_outreach_companies)
outreach_save_button.on_click(save_outreach_activity)

outreach_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Log Outreach Activity</h2>"
    ),
    outreach_company,
    outreach_contact,
    outreach_refresh_button,
    outreach_activity_type,
    outreach_direction,
    outreach_outcome,
    outreach_date,
    outreach_notes,
    outreach_next_action,
    outreach_followup_date,
    outreach_save_button,
    outreach_output
])

# Register with the CRM menu

# Display the refreshed interactive menu


In [ ]:
# STEP 039 — Schedule follow-up form

# NORTH STAR ANALYTICS — SCHEDULE FOLLOW-UP FORM


fu_style = {"description_width": "130px"}
fu_width = widgets.Layout(width="620px")

def fu_company_options():
    options = [("Select a company...", "")]
    for r in companies_sheet.get_all_records():
        cid = str(r.get("Company ID", "")).strip()
        name = str(r.get("Company Name", "")).strip()
        if cid and name:
            options.append((name, cid))
    return options

fu_company = widgets.Dropdown(
    options=fu_company_options(),
    description="Company:",
    style=fu_style,
    layout=fu_width
)

fu_contact = widgets.Dropdown(
    options=[("No contact", "")],
    description="Contact:",
    style=fu_style,
    layout=fu_width
)

fu_opportunity = widgets.Dropdown(
    options=[("No opportunity", "")],
    description="Opportunity:",
    style=fu_style,
    layout=fu_width
)

def fu_update_related(change=None):
    cid = fu_company.value
    contacts = [("No contact", "")]
    opportunities = [("No opportunity", "")]

    if cid:
        for r in contacts_sheet.get_all_records():
            if str(r.get("Company ID", "")).strip() == cid:
                contact_id = str(r.get("Contact ID", "")).strip()
                name = (
                    str(r.get("First Name", "")).strip() + " " +
                    str(r.get("Last Name", "")).strip()
                ).strip()
                if contact_id:
                    contacts.append((name or contact_id, contact_id))

        for r in opportunities_sheet.get_all_records():
            if str(r.get("Company ID", "")).strip() == cid:
                oid = str(r.get("Opportunity ID", "")).strip()
                name = str(
                    r.get("Opportunity Name")
                    or r.get("Opportunity Title")
                    or oid
                ).strip()
                if oid:
                    opportunities.append((name, oid))

    fu_contact.options = contacts
    fu_opportunity.options = opportunities

fu_company.observe(fu_update_related, names="value")

fu_description = widgets.Text(
    description="Task:",
    placeholder="Call the general manager about discovery...",
    style=fu_style,
    layout=fu_width
)

fu_due_date = widgets.DatePicker(
    description="Due Date:",
    value=date.today(),
    style=fu_style
)

fu_task_type = widgets.Dropdown(
    options=[
        "Follow-Up", "Phone", "Email", "LinkedIn",
        "Meeting", "Discovery", "Proposal", "Other"
    ],
    description="Task Type:",
    style=fu_style
)

fu_priority = widgets.Dropdown(
    options=["High", "Medium", "Low"],
    value="Medium",
    description="Priority:",
    style=fu_style
)

fu_notes = widgets.Textarea(
    description="Notes:",
    style=fu_style,
    layout=widgets.Layout(width="620px", height="100px")
)

fu_refresh = widgets.Button(
    description="Refresh Companies",
    icon="refresh"
)

fu_save = widgets.Button(
    description="Schedule Follow-Up",
    button_style="success",
    icon="calendar-plus"
)

fu_output = widgets.Output()

def fu_refresh_data(_):
    with fu_output:
        clear_output()
        try:
            current = fu_company.value
            fu_company.options = fu_company_options()
            if current in [v for _, v in fu_company.options]:
                fu_company.value = current
            fu_update_related()
            print("✓ Company, contact, and opportunity lists refreshed.")
        except Exception as error:
            print("Unable to refresh:", error)

def fu_save_task(_):
    with fu_output:
        clear_output()

        if not fu_company.value:
            print("⚠ Select a company.")
            return

        if not fu_description.value.strip():
            print("⚠ Enter a task description.")
            return

        if not fu_due_date.value:
            print("⚠ Select a due date.")
            return

        fu_save.disabled = True

        try:
            result = create_followup(
                company_id=fu_company.value,
                task_description=fu_description.value.strip(),
                due_date=fu_due_date.value.isoformat(),
                contact_id=fu_contact.value or "",
                opportunity_id=fu_opportunity.value or "",
                task_type=fu_task_type.value,
                priority=fu_priority.value,
                notes=fu_notes.value.strip()
            )

            print("✓ Follow-up scheduled successfully.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print("Check the Follow-Ups worksheet before retrying.")

        finally:
            fu_save.disabled = False

fu_refresh.on_click(fu_refresh_data)
fu_save.on_click(fu_save_task)

fu_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Schedule Follow-Up</h2>"),
    fu_company,
    fu_contact,
    fu_opportunity,
    fu_refresh,
    fu_description,
    fu_due_date,
    fu_task_type,
    fu_priority,
    fu_notes,
    fu_save,
    fu_output
])

# Register the form in the existing CRM menu

# Reopen the updated interactive menu


In [ ]:
# STEP 040 — Follow-up task manager

# NORTH STAR ANALYTICS — VIEW & COMPLETE FOLLOW-UPS


task_style = {"description_width": "130px"}
task_width = widgets.Layout(width="750px")

task_selector = widgets.Dropdown(
    options=[("Loading tasks...", "")],
    description="Open Task:",
    style=task_style,
    layout=task_width
)

task_details = widgets.Output()
task_message = widgets.Output()

task_refresh_button = widgets.Button(
    description="Refresh Tasks",
    icon="refresh"
)

task_complete_button = widgets.Button(
    description="Mark Task Complete",
    button_style="success",
    icon="check-circle"
)

def load_open_followups():
    records = followups_sheet.get_all_records()

    company_lookup = {
        str(r.get("Company ID", "")).strip():
        str(r.get("Company Name", "")).strip()
        for r in companies_sheet.get_all_records()
    }

    open_tasks = []

    for r in records:
        task_id = str(r.get("Task ID", "")).strip()
        status = str(r.get("Status", "")).strip().lower()

        if not task_id:
            continue

        if status in ("completed", "complete", "done", "cancelled", "canceled"):
            continue

        open_tasks.append(r)

    def sort_key(r):
        return str(r.get("Due Date", "9999-12-31"))

    open_tasks.sort(key=sort_key)

    options = [("Select a follow-up...", "")]

    for r in open_tasks:
        cid = str(r.get("Company ID", "")).strip()
        company = company_lookup.get(cid, cid)
        description = str(r.get("Task Description", "")).strip()
        due = str(r.get("Due Date", "")).strip()
        priority = str(r.get("Priority", "")).strip()
        task_id = str(r.get("Task ID", "")).strip()

        label = f"{due} | {priority} | {company} | {description}"
        options.append((label[:140], task_id))

    return options, open_tasks, company_lookup

task_records = []
task_company_lookup = {}

def refresh_task_list(_=None):
    global task_records, task_company_lookup

    with task_message:
        clear_output()

        try:
            options, task_records, task_company_lookup = load_open_followups()
            task_selector.options = options

            print(f"✓ {len(task_records)} outstanding follow-up(s) loaded.")

        except Exception as error:
            print("Unable to load tasks:", error)

    show_selected_task()

def show_selected_task(change=None):
    with task_details:
        clear_output()

        selected_id = task_selector.value

        if not selected_id:
            print("Select a task to view its details.")
            return

        task = next(
            (
                r for r in task_records
                if str(r.get("Task ID", "")).strip() == selected_id
            ),
            None
        )

        if task is None:
            print("Task not found. Refresh the list.")
            return

        cid = str(task.get("Company ID", "")).strip()

        print("=" * 55)
        print("FOLLOW-UP TASK DETAILS")
        print("=" * 55)
        print("Company:", task_company_lookup.get(cid, cid))
        print("Task:", task.get("Task Description", ""))
        print("Type:", task.get("Task Type", ""))
        print("Priority:", task.get("Priority", ""))
        print("Due Date:", task.get("Due Date", ""))
        print("Status:", task.get("Status", ""))
        print("Notes:", task.get("Notes", ""))

def complete_selected_task(_):
    selected_id = task_selector.value

    with task_message:
        clear_output()

        if not selected_id:
            print("⚠ Select a task first.")
            return

        task_complete_button.disabled = True

        try:
            result = complete_followup(selected_id)
            print("✓ Completion function executed.")
            print("Result:", result)

            # Reload directly from Google Sheets to verify completion
            refresh_task_list()

        except Exception as error:
            print("⚠ Unable to confirm completion:", error)
            print("Check the Follow-Ups worksheet before retrying.")

        finally:
            task_complete_button.disabled = False

task_selector.observe(show_selected_task, names="value")
task_refresh_button.on_click(refresh_task_list)
task_complete_button.on_click(complete_selected_task)

task_management_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Follow-Up Task Manager</h2>"),
    widgets.HBox([task_refresh_button, task_complete_button]),
    task_selector,
    task_details,
    task_message
])

# Register in the CRM menu


# Display the updated CRM menu

print("✓ No follow-up records modified.")


In [ ]:
# STEP 041 — New opportunity form

# NORTH STAR ANALYTICS — ADD NEW OPPORTUNITY


opp_style = {"description_width": "145px"}
opp_width = widgets.Layout(width="650px")

def opp_company_options():
    options = [("Select a company...", "")]
    for r in companies_sheet.get_all_records():
        cid = str(r.get("Company ID", "")).strip()
        name = str(r.get("Company Name", "")).strip()
        if cid and name:
            options.append((name, cid))
    return options

opp_company = widgets.Dropdown(
    options=opp_company_options(),
    description="Company:",
    style=opp_style,
    layout=opp_width
)

opp_contact = widgets.Dropdown(
    options=[("No contact selected", "")],
    description="Contact:",
    style=opp_style,
    layout=opp_width
)

def opp_update_contacts(change=None):
    cid = opp_company.value
    options = [("No contact selected", "")]

    if cid:
        for r in contacts_sheet.get_all_records():
            if str(r.get("Company ID", "")).strip() == cid:
                contact_id = str(r.get("Contact ID", "")).strip()
                name = (
                    str(r.get("First Name", "")).strip() + " " +
                    str(r.get("Last Name", "")).strip()
                ).strip()
                if contact_id:
                    options.append((name or contact_id, contact_id))

    opp_contact.options = options

opp_company.observe(opp_update_contacts, names="value")

opp_name = widgets.Text(
    description="Opportunity Name:",
    placeholder="Dealership Sales Process Assessment",
    style=opp_style,
    layout=opp_width
)

opp_service = widgets.Dropdown(
    options=[
        "Operational Discovery Assessment",
        "Business Process Improvement",
        "Sales Performance Analysis",
        "KPI Dashboard & Reporting",
        "Data Quality Assessment",
        "Workflow Automation",
        "Employee Training",
        "Continuing Analytics Services",
        "Custom Consulting Package",
        "Other"
    ],
    description="Service Interest:",
    style=opp_style,
    layout=opp_width
)

opp_value = widgets.FloatText(
    value=0,
    description="Estimated Value ($):",
    style=opp_style
)

opp_stage = widgets.Dropdown(
    options=[
        "Target identified",
        "First contact attempted",
        "Stakeholder identified",
        "Conversation scheduled",
        "Discovery completed",
        "Qualified opportunity",
        "Proposal sent",
        "Decision pending",
        "Won",
        "Lost",
        "Nurture",
        "Project active"
    ],
    description="Pipeline Stage:",
    style=opp_style,
    layout=opp_width
)

opp_close_date = widgets.DatePicker(
    description="Expected Close:",
    style=opp_style
)

opp_tags = widgets.Text(
    description="Tags:",
    placeholder="Dealership, High Priority, Sales",
    style=opp_style,
    layout=opp_width
)

opp_notes = widgets.Textarea(
    description="Notes:",
    placeholder="Describe the business need, stakeholders, and opportunity...",
    style=opp_style,
    layout=widgets.Layout(width="650px", height="110px")
)

opp_refresh = widgets.Button(
    description="Refresh Companies",
    icon="refresh"
)

opp_save = widgets.Button(
    description="Save Opportunity",
    button_style="success",
    icon="save"
)

opp_output = widgets.Output()

def opp_refresh_data(_):
    with opp_output:
        clear_output()
        try:
            current = opp_company.value
            opp_company.options = opp_company_options()

            if current in [v for _, v in opp_company.options]:
                opp_company.value = current

            opp_update_contacts()
            print("✓ Company and contact lists refreshed.")
        except Exception as error:
            print("Unable to refresh:", error)

def opp_save_record(_):
    with opp_output:
        clear_output()

        if not opp_company.value:
            print("⚠ Select a company.")
            return

        if not opp_name.value.strip():
            print("⚠ Enter an opportunity name.")
            return

        if opp_value.value < 0:
            print("⚠ Estimated value cannot be negative.")
            return

        opp_save.disabled = True

        try:
            result = create_opportunity(
                company_id=opp_company.value,
                opportunity_name=opp_name.value.strip(),
                service_interest=opp_service.value,
                estimated_value=opp_value.value,
                contact_id=opp_contact.value or "",
                pipeline_stage=opp_stage.value,
                tags=[
                    tag.strip()
                    for tag in opp_tags.value.split(",")
                    if tag.strip()
                ],
                expected_close_date=(
                    opp_close_date.value.isoformat()
                    if opp_close_date.value else ""
                ),
                notes=opp_notes.value.strip()
            )

            print("✓ Opportunity saved successfully.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print(
                "Check the Opportunities worksheet "
                "before retrying to avoid duplicates."
            )

        finally:
            opp_save.disabled = False

opp_refresh.on_click(opp_refresh_data)
opp_save.on_click(opp_save_record)

opp_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Add Opportunity</h2>"),
    opp_company,
    opp_contact,
    opp_refresh,
    opp_name,
    opp_service,
    opp_value,
    opp_stage,
    opp_close_date,
    opp_tags,
    opp_notes,
    opp_save,
    opp_output
])

# Register in the existing interactive menu

# Display the refreshed menu

print("✓ No CRM records modified.")


In [ ]:
# STEP 042 — Opportunity stage manager

# NORTH STAR ANALYTICS — PIPELINE STAGE MANAGER


stage_style = {"description_width": "145px"}
stage_width = widgets.Layout(width="720px")

stage_options = [
    "Target identified",
    "First contact attempted",
    "Stakeholder identified",
    "Conversation scheduled",
    "Discovery completed",
    "Qualified opportunity",
    "Proposal sent",
    "Decision pending",
    "Won",
    "Lost",
    "Nurture",
    "Project active"
]

stage_opportunity = widgets.Dropdown(
    options=[("Select an opportunity...", "")],
    description="Opportunity:",
    style=stage_style,
    layout=stage_width
)

stage_new = widgets.Dropdown(
    options=stage_options,
    description="New Stage:",
    style=stage_style,
    layout=stage_width
)

stage_details = widgets.Output()
stage_message = widgets.Output()

stage_refresh = widgets.Button(
    description="Refresh Opportunities",
    icon="refresh"
)

stage_save = widgets.Button(
    description="Update Pipeline Stage",
    button_style="success",
    icon="check"
)

stage_records = {}
stage_companies = {}

def stage_load_opportunities(_=None):
    global stage_records, stage_companies

    with stage_message:
        clear_output()

        try:
            stage_companies = {
                str(r.get("Company ID", "")).strip():
                str(r.get("Company Name", "")).strip()
                for r in companies_sheet.get_all_records()
            }

            stage_records = {
                str(r.get("Opportunity ID", "")).strip(): r
                for r in opportunities_sheet.get_all_records()
                if str(r.get("Opportunity ID", "")).strip()
            }

            options = [("Select an opportunity...", "")]

            for oid, r in stage_records.items():
                cid = str(r.get("Company ID", "")).strip()
                company = stage_companies.get(cid, cid)
                name = str(r.get("Opportunity Name", "")).strip()
                current = str(r.get("Pipeline Stage", "")).strip()

                options.append((
                    f"{company} | {name} | {current}",
                    oid
                ))

            stage_opportunity.options = options
            print(f"✓ {len(stage_records)} opportunities loaded.")

        except Exception as error:
            print("Unable to load opportunities:", error)

def stage_show_details(change=None):
    with stage_details:
        clear_output()

        oid = stage_opportunity.value
        if not oid or oid not in stage_records:
            print("Select an opportunity to view its current stage.")
            return

        r = stage_records[oid]
        cid = str(r.get("Company ID", "")).strip()

        print("=" * 55)
        print("OPPORTUNITY DETAILS")
        print("=" * 55)
        print("Company:", stage_companies.get(cid, cid))
        print("Opportunity:", r.get("Opportunity Name", ""))
        print("Current Stage:", r.get("Pipeline Stage", ""))
        print("Estimated Value:", r.get("Estimated Value", ""))
        print("Opportunity Score:", r.get("Opportunity Score", ""))
        print("Expected Close:", r.get("Expected Close Date", ""))

        current_stage = str(r.get("Pipeline Stage", "")).strip()
        if current_stage in stage_options:
            stage_new.value = current_stage

def stage_update_record(_):
    oid = stage_opportunity.value

    with stage_message:
        clear_output()

        if not oid:
            print("⚠ Select an opportunity.")
            return

        current_stage = str(
            stage_records.get(oid, {}).get("Pipeline Stage", "")
        ).strip()

        if stage_new.value == current_stage:
            print("No change needed. This opportunity is already in that stage.")
            return

        stage_save.disabled = True

        try:
            result = update_opportunity_stage(
                opportunity_id=oid,
                new_stage=stage_new.value
            )

            print("✓ Pipeline stage update function executed.")
            print("Result:", result)

            # Reload current values from Google Sheets
            stage_load_opportunities()

        except Exception as error:
            print("⚠ Unable to confirm update:", error)
            print("Check the Opportunities worksheet before retrying.")

        finally:
            stage_save.disabled = False

stage_opportunity.observe(stage_show_details, names="value")
stage_refresh.on_click(stage_load_opportunities)
stage_save.on_click(stage_update_record)

stage_form = widgets.VBox([
    widgets.HTML("<h2>North Star Analytics — Pipeline Stage Manager</h2>"),
    stage_refresh,
    stage_opportunity,
    stage_details,
    stage_new,
    stage_save,
    stage_message
])

# Register in CRM menu


# Display refreshed CRM menu

print("✓ No opportunity records modified.")


In [ ]:
# STEP 043 — Opportunity scoring form

# NORTH STAR ANALYTICS — OPPORTUNITY SCORING FORM


score_style = {"description_width": "170px"}
score_width = widgets.Layout(width="700px")

score_opportunity_selector = widgets.Dropdown(
    options=[("Select an opportunity...", "")],
    description="Opportunity:",
    style=score_style,
    layout=score_width
)

score_refresh = widgets.Button(
    description="Refresh Opportunities",
    icon="refresh"
)

score_criteria = {
    "Business Need": "How clearly does the prospect need improvement?",
    "Decision-Maker Access": "Can you reach someone authorized to approve services?",
    "Budget Readiness": "How prepared is the prospect to fund the work?",
    "Timeline Urgency": "How soon does the prospect need a solution?",
    "Engagement Level": "How actively is the prospect communicating?",
    "Service Fit": "How closely do their needs match your services?"
}

score_sliders = {}

for criterion, explanation in score_criteria.items():
    score_sliders[criterion] = widgets.IntSlider(
        value=0,
        min=0,
        max=5,
        step=1,
        description=criterion + ":",
        style=score_style,
        layout=score_width,
        continuous_update=False
    )

score_preview = widgets.HTML()
score_output = widgets.Output()

score_save = widgets.Button(
    description="Save Opportunity Score",
    button_style="success",
    icon="save"
)

def score_classification(value):
    if value >= 80:
        return "High Priority"
    elif value >= 60:
        return "Qualified Prospect"
    elif value >= 40:
        return "Developing Opportunity"
    return "Low Priority / Nurture"

def calculate_score_preview(change=None):
    try:
        ratings = {
            name: slider.value
            for name, slider in score_sliders.items()
        }

        total = round(sum(
            (ratings[name] / 5) * scoring_weights[name]
            for name in ratings
        ), 2)

        classification = score_classification(total)

        score_preview.value = (
            f"<h3>Projected Score: {total}/100</h3>"
            f"<p><b>Classification:</b> {classification}</p>"
            "<p><small>Preview only — not saved yet.</small></p>"
        )

    except Exception as error:
        score_preview.value = (
            f"<p>Unable to calculate preview: {error}</p>"
        )

for slider in score_sliders.values():
    slider.observe(calculate_score_preview, names="value")

def score_load_opportunities(_=None):
    with score_output:
        clear_output()

        try:
            company_lookup = {
                str(r.get("Company ID", "")).strip():
                str(r.get("Company Name", "")).strip()
                for r in companies_sheet.get_all_records()
            }

            options = [("Select an opportunity...", "")]

            for record in opportunities_sheet.get_all_records():
                oid = str(record.get("Opportunity ID", "")).strip()
                if not oid:
                    continue

                cid = str(record.get("Company ID", "")).strip()
                company = company_lookup.get(cid, cid)
                name = str(record.get("Opportunity Name", "")).strip()

                options.append((
                    f"{company} | {name}",
                    oid
                ))

            score_opportunity_selector.options = options
            print(f"✓ {len(options) - 1} opportunities loaded.")

        except Exception as error:
            print("Unable to load opportunities:", error)

def save_opportunity_score(_):
    with score_output:
        clear_output()

        oid = score_opportunity_selector.value

        if not oid:
            print("⚠ Select an opportunity.")
            return

        score_save.disabled = True

        try:
            result = score_opportunity(
                opportunity_id=oid,
                business_need=score_sliders["Business Need"].value,
                decision_maker_access=score_sliders[
                    "Decision-Maker Access"
                ].value,
                budget_readiness=score_sliders["Budget Readiness"].value,
                timeline_urgency=score_sliders["Timeline Urgency"].value,
                engagement_level=score_sliders["Engagement Level"].value,
                service_fit=score_sliders["Service Fit"].value
            )

            print("✓ Opportunity scoring completed.")
            print("Score:", result["score"], "/ 100")
            print("Classification:", result["classification"])

        except Exception as error:
            print("⚠ Unable to confirm score:", error)
            print("Check the Opportunities worksheet before retrying.")

        finally:
            score_save.disabled = False

score_refresh.on_click(score_load_opportunities)
score_save.on_click(save_opportunity_score)

score_form_items = [
    widgets.HTML(
        "<h2>North Star Analytics — Opportunity Scoring</h2>"
    ),
    widgets.HTML(
        "<p>Rate each criterion from 0 (lowest) to 5 (highest).</p>"
    ),
    score_refresh,
    score_opportunity_selector
]

for criterion, slider in score_sliders.items():
    score_form_items.append(
        widgets.HTML(
            f"<b>{criterion}</b><br>"
            f"<small>{score_criteria[criterion]}</small>"
        )
    )
    score_form_items.append(slider)

score_form_items.extend([
    score_preview,
    score_save,
    score_output
])

score_form = widgets.VBox(score_form_items)

calculate_score_preview()

# Register with the existing CRM menu


# Display updated CRM menu

print("✓ No opportunity scores modified.")


In [ ]:
# STEP 044 — Discovery interview creation form

# NORTH STAR ANALYTICS — CREATE DISCOVERY INTERVIEW FORM


di_style = {"description_width": "145px"}
di_width = widgets.Layout(width="680px")

di_company = widgets.Dropdown(
    options=[("Select a company...", "")],
    description="Company:",
    style=di_style,
    layout=di_width
)

di_contact = widgets.Dropdown(
    options=[("No contact selected", "")],
    description="Contact:",
    style=di_style,
    layout=di_width
)

di_department = widgets.Dropdown(
    options=[("Select a department...", "")],
    description="Department:",
    style=di_style,
    layout=di_width
)

di_refresh = widgets.Button(
    description="Refresh Lists",
    icon="refresh"
)

di_create = widgets.Button(
    description="Create Discovery Interview",
    button_style="success",
    icon="clipboard"
)

di_output = widgets.Output()

def di_update_contacts(change=None):
    cid = di_company.value
    options = [("No contact selected", "")]

    if cid:
        for r in contacts_sheet.get_all_records():
            if str(r.get("Company ID", "")).strip() == cid:
                contact_id = str(r.get("Contact ID", "")).strip()
                name = (
                    str(r.get("First Name", "")).strip() + " " +
                    str(r.get("Last Name", "")).strip()
                ).strip()

                if contact_id:
                    options.append((name or contact_id, contact_id))

    di_contact.options = options

di_company.observe(di_update_contacts, names="value")

def di_refresh_lists(_=None):
    with di_output:
        clear_output()

        try:
            companies = [
                (
                    str(r.get("Company Name", "")).strip(),
                    str(r.get("Company ID", "")).strip()
                )
                for r in companies_sheet.get_all_records()
                if r.get("Company ID")
            ]

            di_company.options = [
                ("Select a company...", "")
            ] + companies

            departments = sorted({
                str(r.get("Department", "")).strip()
                for r in question_library_sheet.get_all_records()
                if str(r.get("Department", "")).strip()
            })

            di_department.options = [
                ("Select a department...", "")
            ] + [(d, d) for d in departments]

            di_update_contacts()

            print(f"✓ {len(companies)} companies loaded.")
            print(f"✓ {len(departments)} departments loaded.")

        except Exception as error:
            print("Unable to refresh lists:", error)

def di_create_interview(_):
    with di_output:
        clear_output()

        if not di_company.value:
            print("⚠ Select a company.")
            return

        if not di_department.value:
            print("⚠ Select a department.")
            return

        di_create.disabled = True

        try:
            interview_id = create_discovery_interview(
                company_id=di_company.value,
                department=di_department.value,
                contact_id=di_contact.value or ""
            )

            print("\n✓ Interview created successfully.")
            print("Interview ID:", interview_id)
            print(
                "The department questions are now stored "
                "in your Discovery Interviews worksheet."
            )

        except Exception as error:
            print("⚠ Unable to confirm interview creation:", error)
            print(
                "Check the Discovery Interviews worksheet "
                "before retrying to avoid duplicates."
            )

        finally:
            di_create.disabled = False

di_refresh.on_click(di_refresh_lists)
di_create.on_click(di_create_interview)

di_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Create Discovery Interview</h2>"
    ),
    di_refresh,
    di_company,
    di_contact,
    di_department,
    di_create,
    di_output
])

# Register in existing CRM menu


# Display updated menu

print("✓ No interview records created during setup.")


In [ ]:
# STEP 045 — Discovery response manager

# NORTH STAR ANALYTICS — DISCOVERY INTERVIEW RESPONSE MANAGER


ir_style = {"description_width": "150px"}
ir_width = widgets.Layout(width="750px")

ir_interview = widgets.Dropdown(
    options=[("Select an interview...", "")],
    description="Interview:",
    style=ir_style,
    layout=ir_width
)

ir_question = widgets.Dropdown(
    options=[("Select a question...", "")],
    description="Question:",
    style=ir_style,
    layout=ir_width
)

ir_business_question = widgets.HTML()

def ir_textarea(label, placeholder=""):
    return widgets.Textarea(
        description=label,
        placeholder=placeholder,
        style=ir_style,
        layout=widgets.Layout(width="750px", height="90px")
    )

ir_response = ir_textarea("Client Response:")
ir_pain = ir_textarea("Pain Point:")
ir_impact = ir_textarea("Business Impact:")
ir_data = ir_textarea("Available Data:")
ir_action = ir_textarea("Follow-Up Action:")

ir_status = widgets.Dropdown(
    options=["Not Started", "In Progress", "Completed"],
    description="Status:",
    style=ir_style
)

ir_refresh = widgets.Button(
    description="Refresh Interviews",
    icon="refresh"
)

ir_save = widgets.Button(
    description="Save Question Response",
    button_style="success",
    icon="save"
)

ir_output = widgets.Output()

ir_records = []
ir_selected_row = None

def ir_load_interviews(_=None):
    global ir_records

    with ir_output:
        clear_output()

        try:
            ir_records = interview_sheet.get_all_records()

            companies = {
                str(r.get("Company ID", "")).strip():
                str(r.get("Company Name", "")).strip()
                for r in companies_sheet.get_all_records()
            }

            interviews = {}

            for r in ir_records:
                iid = str(r.get("Interview ID", "")).strip()
                if iid and iid not in interviews:
                    interviews[iid] = r

            options = [("Select an interview...", "")]

            for iid, r in interviews.items():
                cid = str(r.get("Company ID", "")).strip()
                company = companies.get(cid, cid)
                department = str(r.get("Department", "")).strip()

                options.append((
                    f"{company} | {department} | {iid}",
                    iid
                ))

            ir_interview.options = options

            print(f"✓ {len(interviews)} interviews loaded.")

        except Exception as error:
            print("Unable to load interviews:", error)

def ir_load_questions(change=None):
    iid = ir_interview.value

    options = [("Select a question...", "")]

    if iid:
        for index, r in enumerate(ir_records):
            if str(r.get("Interview ID", "")).strip() == iid:
                qid = str(r.get("Question ID", "")).strip()
                question = str(r.get("Business Question", "")).strip()

                options.append((
                    f"{qid} | {question[:80]}",
                    index
                ))

    ir_question.options = options

def ir_load_response(change=None):
    global ir_selected_row

    ir_selected_row = None

    index = ir_question.value

    if index == "" or index is None:
        ir_business_question.value = ""
        return

    r = ir_records[index]
    ir_selected_row = index + 2

    ir_business_question.value = (
        "<h3>Discovery Question</h3>"
        f"<p>{r.get('Business Question', '')}</p>"
    )

    ir_response.value = str(r.get("Client Response", ""))
    ir_pain.value = str(r.get("Identified Pain Point", ""))
    ir_impact.value = str(r.get("Business Impact", ""))
    ir_data.value = str(r.get("Available Data", ""))
    ir_action.value = str(r.get("Follow-Up Action", ""))

    status = str(r.get("Status", "Not Started")).strip()
    ir_status.value = (
        status if status in ir_status.options else "Not Started"
    )

def ir_save_response(_):
    with ir_output:
        clear_output()

        if ir_selected_row is None:
            print("⚠ Select an interview question.")
            return

        ir_save.disabled = True

        try:
            row = ir_selected_row

            # Columns H through M:
            # Client Response, Pain Point, Business Impact,
            # Available Data, Follow-Up Action, Status

            interview_sheet.update(
                range_name=f"H{row}:M{row}",
                values=[[
                    ir_response.value.strip(),
                    ir_pain.value.strip(),
                    ir_impact.value.strip(),
                    ir_data.value.strip(),
                    ir_action.value.strip(),
                    ir_status.value
                ]],
                value_input_option="RAW"
            )

            print("✓ Discovery response saved successfully.")
            print("Interview:", ir_interview.value)
            print("Question:", ir_question.label)

            # Refresh local copy
            ir_records[row - 2].update({
                "Client Response": ir_response.value.strip(),
                "Identified Pain Point": ir_pain.value.strip(),
                "Business Impact": ir_impact.value.strip(),
                "Available Data": ir_data.value.strip(),
                "Follow-Up Action": ir_action.value.strip(),
                "Status": ir_status.value
            })

        except Exception as error:
            print("⚠ Unable to confirm save:", error)
            print("Check the Discovery Interviews worksheet.")

        finally:
            ir_save.disabled = False

ir_interview.observe(ir_load_questions, names="value")
ir_question.observe(ir_load_response, names="value")
ir_refresh.on_click(ir_load_interviews)
ir_save.on_click(ir_save_response)

ir_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Record Discovery Responses</h2>"
    ),
    ir_refresh,
    ir_interview,
    ir_question,
    ir_business_question,
    ir_response,
    ir_pain,
    ir_impact,
    ir_data,
    ir_action,
    ir_status,
    ir_save,
    ir_output
])

# Register in existing CRM menu


# Display refreshed menu

print("✓ No interview responses modified during setup.")


In [ ]:
# STEP 046 — Discovery findings manager

# NORTH STAR ANALYTICS — DISCOVERY FINDINGS MANAGER


df_style = {"description_width": "150px"}
df_width = widgets.Layout(width="750px")

df_interview = widgets.Dropdown(
    options=[("Select an interview...", "")],
    description="Interview:",
    style=df_style,
    layout=df_width
)

df_refresh = widgets.Button(
    description="Refresh Interviews",
    icon="refresh"
)

df_view = widgets.Button(
    description="View Findings",
    icon="eye"
)

df_transfer = widgets.Button(
    description="Transfer Interview Findings",
    button_style="warning",
    icon="arrow-right"
)

df_recommend = widgets.Button(
    description="Generate Recommendations",
    button_style="success",
    icon="lightbulb"
)

df_output = widgets.Output()
df_interview_records = {}

def df_load_interviews(_=None):
    global df_interview_records

    with df_output:
        clear_output()

        try:
            companies = {
                str(r.get("Company ID", "")).strip():
                str(r.get("Company Name", "")).strip()
                for r in companies_sheet.get_all_records()
            }

            df_interview_records = {}

            for r in interview_sheet.get_all_records():
                iid = str(r.get("Interview ID", "")).strip()
                if iid and iid not in df_interview_records:
                    df_interview_records[iid] = r

            options = [("Select an interview...", "")]

            for iid, r in df_interview_records.items():
                cid = str(r.get("Company ID", "")).strip()
                company = companies.get(cid, cid)
                department = str(r.get("Department", "")).strip()

                options.append((
                    f"{company} | {department} | {iid}",
                    iid
                ))

            df_interview.options = options
            print(f"✓ {len(df_interview_records)} interviews loaded.")

        except Exception as error:
            print("Unable to load interviews:", error)

def df_view_findings(_):
    with df_output:
        clear_output()

        if not df_interview.value:
            print("⚠ Select an interview.")
            return

        try:
            show_interview_findings(df_interview.value)
        except Exception as error:
            print("Unable to display findings:", error)

def df_transfer_findings(_):
    with df_output:
        clear_output()

        if not df_interview.value:
            print("⚠ Select an interview.")
            return

        df_transfer.disabled = True

        try:
            result = transfer_interview_findings(
                df_interview.value
            )

            print("✓ Findings transfer function completed.")
            print("Result:", result)

        except Exception as error:
            print("⚠ Unable to confirm transfer:", error)
            print(
                "Check the Discovery worksheet before retrying "
                "to avoid duplicate findings."
            )

        finally:
            df_transfer.disabled = False

def df_generate_recommendations(_):
    with df_output:
        clear_output()

        iid = df_interview.value

        if not iid:
            print("⚠ Select an interview.")
            return

        interview = df_interview_records.get(iid)

        if not interview:
            print("⚠ Interview not found. Refresh the list.")
            return

        company_id = str(interview.get("Company ID", "")).strip()

        df_recommend.disabled = True

        try:
            result = generate_consulting_recommendations(
                company_id
            )

            if result is not None:
                display(result)

        except Exception as error:
            print("Unable to generate recommendations:", error)

        finally:
            df_recommend.disabled = False

df_refresh.on_click(df_load_interviews)
df_view.on_click(df_view_findings)
df_transfer.on_click(df_transfer_findings)
df_recommend.on_click(df_generate_recommendations)

df_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Discovery Findings Manager</h2>"
    ),
    df_refresh,
    df_interview,
    widgets.HBox([
        df_view,
        df_transfer,
        df_recommend
    ]),
    df_output
])

# Register with existing CRM menu


# Display updated CRM menu

print("✓ No discovery records modified during setup.")


In [ ]:
# STEP 047 — Proposal scope draft form

# NORTH STAR ANALYTICS — PROPOSAL SCOPE DRAFT MANAGER


ps_style = {"description_width": "150px"}
ps_width = widgets.Layout(width="750px")

ps_company = widgets.Dropdown(
    options=[("Select a company...", "")],
    description="Company:",
    style=ps_style,
    layout=ps_width
)

ps_existing = widgets.Dropdown(
    options=[("Select an existing draft...", "")],
    description="Scope Draft:",
    style=ps_style,
    layout=ps_width
)

ps_refresh = widgets.Button(
    description="Refresh Lists",
    icon="refresh"
)

ps_create = widgets.Button(
    description="Generate Scope Draft",
    button_style="success",
    icon="file-plus"
)

ps_view = widgets.Button(
    description="View Existing Draft",
    icon="eye"
)

ps_output = widgets.Output()

ps_scope_records = {}

def ps_refresh_lists(_=None):
    global ps_scope_records

    with ps_output:
        clear_output()

        try:
            company_options = [("Select a company...", "")]
            company_lookup = {}

            for r in companies_sheet.get_all_records():
                cid = str(r.get("Company ID", "")).strip()
                name = str(r.get("Company Name", "")).strip()

                if cid:
                    company_options.append((name or cid, cid))
                    company_lookup[cid] = name or cid

            ps_company.options = company_options

            ps_scope_records = {}
            draft_options = [("Select an existing draft...", "")]

            for r in scope_sheet.get_all_records():
                sid = str(r.get("Scope Draft ID", "")).strip()
                if not sid:
                    continue

                ps_scope_records[sid] = r

                cid = str(r.get("Company ID", "")).strip()
                company = company_lookup.get(cid, cid)
                status = str(r.get("Draft Status", "")).strip()

                draft_options.append((
                    f"{company} | {status} | {sid}",
                    sid
                ))

            ps_existing.options = draft_options

            print(f"✓ {len(company_lookup)} companies loaded.")
            print(f"✓ {len(ps_scope_records)} scope drafts loaded.")

        except Exception as error:
            print("Unable to refresh:", error)

def ps_create_draft(_):
    with ps_output:
        clear_output()

        if not ps_company.value:
            print("⚠ Select a company.")
            return

        ps_create.disabled = True

        try:
            result = create_proposal_scope_draft(
                company_id=ps_company.value
            )

            print("✓ Scope draft generation completed.")
            print("Result:", result)
            print(
                "Review the new draft before approving pricing."
            )

        except Exception as error:
            print("⚠ Unable to confirm draft creation:", error)
            print(
                "Check the Proposal Scope Drafts worksheet "
                "before retrying to avoid duplicates."
            )

        finally:
            ps_create.disabled = False

def ps_view_draft(_):
    with ps_output:
        clear_output()

        if not ps_existing.value:
            print("⚠ Select an existing scope draft.")
            return

        try:
            show_proposal_scope_draft(ps_existing.value)
        except Exception as error:
            print("Unable to display scope draft:", error)

ps_refresh.on_click(ps_refresh_lists)
ps_create.on_click(ps_create_draft)
ps_view.on_click(ps_view_draft)

ps_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Proposal Scope Draft Manager</h2>"
    ),
    ps_refresh,
    ps_company,
    ps_create,
    widgets.HTML("<hr><h3>Existing Scope Drafts</h3>"),
    ps_existing,
    ps_view,
    ps_output
])

# Register in the existing CRM menu


# Display refreshed menu

print("✓ No proposal records modified during setup.")


In [ ]:
# STEP 048 — Proposal review and approval form

# Original section: PROPOSAL REVIEW & APPROVAL MANAGER
# NORTH STAR ANALYTICS CRM


pr_style = {"description_width": "150px"}
pr_layout = widgets.Layout(width="750px")

pr_proposal = widgets.Dropdown(
    options=[("Select a proposal...", "")],
    description="Proposal:",
    style=pr_style,
    layout=pr_layout
)

pr_check_labels = [
    ("Scope", "scope_confirmed"),
    ("Deliverables", "deliverables_confirmed"),
    ("Timeline", "timeline_confirmed"),
    ("Pricing", "pricing_confirmed"),
    ("Payment Terms", "payment_terms_confirmed"),
    ("Exclusions", "exclusions_confirmed"),
    ("Add-Ons", "addons_confirmed"),
    ("Client Details", "client_details_confirmed")
]

pr_checks = {
    key: widgets.Checkbox(
        value=False,
        description=label,
        indent=False,
        layout=widgets.Layout(width="350px")
    )
    for label, key in pr_check_labels
}

pr_notes = widgets.Textarea(
    description="Review Notes:",
    placeholder="Document missing information, revisions, or approval notes...",
    style=pr_style,
    layout=widgets.Layout(width="750px", height="110px")
)

pr_refresh = widgets.Button(
    description="Refresh Proposals",
    icon="refresh"
)

pr_load = widgets.Button(
    description="Load Existing Review",
    icon="folder-open"
)

pr_save = widgets.Button(
    description="Save Proposal Review",
    button_style="success",
    icon="save"
)

pr_output = widgets.Output()

def pr_refresh_proposals(_=None):
    with pr_output:
        clear_output()

        try:
            records = proposals_sheet.get_all_records()

            options = [("Select a proposal...", "")]

            for r in records:
                proposal_id = str(r.get("Proposal ID", "")).strip()
                if not proposal_id:
                    continue

                label = (
                    f"{proposal_id} | "
                    f"{r.get('Service Package', '')} | "
                    f"${r.get('Total Price', '')} | "
                    f"{r.get('Status', '')}"
                )

                options.append((label, proposal_id))

            pr_proposal.options = options

            print(f"✓ {len(options) - 1} proposals loaded.")

        except Exception as error:
            print("Unable to refresh proposals:", error)

def pr_load_review(_):
    with pr_output:
        clear_output()

        proposal_id = pr_proposal.value

        if not proposal_id:
            print("⚠ Select a proposal first.")
            return

        try:
            review = show_proposal_review_status(proposal_id)

            # Reset controls before loading another proposal
            for checkbox in pr_checks.values():
                checkbox.value = False

            pr_notes.value = ""

            if review:
                field_map = {
                    "scope_confirmed": "Scope Confirmed",
                    "deliverables_confirmed": "Deliverables Confirmed",
                    "timeline_confirmed": "Timeline Confirmed",
                    "pricing_confirmed": "Pricing Confirmed",
                    "payment_terms_confirmed": "Payment Terms Confirmed",
                    "exclusions_confirmed": "Exclusions Confirmed",
                    "addons_confirmed": "Add-Ons Confirmed",
                    "client_details_confirmed": "Client Details Confirmed"
                }

                for key, column in field_map.items():
                    pr_checks[key].value = (
                        str(review.get(column, "")).strip().lower()
                        == "yes"
                    )

                pr_notes.value = str(
                    review.get("Reviewer Notes", "")
                )

                print("✓ Existing review loaded.")
            else:
                print("No existing review. Ready for a new review.")

        except Exception as error:
            print("Unable to load review:", error)

def pr_save_review(_):
    with pr_output:
        clear_output()

        proposal_id = pr_proposal.value

        if not proposal_id:
            print("⚠ Select a proposal first.")
            return

        pr_save.disabled = True

        try:
            status = review_proposal(
                proposal_id=proposal_id,
                scope_confirmed=pr_checks["scope_confirmed"].value,
                deliverables_confirmed=pr_checks["deliverables_confirmed"].value,
                timeline_confirmed=pr_checks["timeline_confirmed"].value,
                pricing_confirmed=pr_checks["pricing_confirmed"].value,
                payment_terms_confirmed=pr_checks["payment_terms_confirmed"].value,
                exclusions_confirmed=pr_checks["exclusions_confirmed"].value,
                addons_confirmed=pr_checks["addons_confirmed"].value,
                client_details_confirmed=pr_checks["client_details_confirmed"].value,
                reviewer_notes=pr_notes.value
            )

            print("\n✓ Proposal review saved.")
            print("Review Status:", status)
            print("No proposal sent to a client.")

        except Exception as error:
            print("⚠ Unable to confirm review save:", error)
            print("Check the Proposal Reviews worksheet before retrying.")

        finally:
            pr_save.disabled = False

pr_refresh.on_click(pr_refresh_proposals)
pr_load.on_click(pr_load_review)
pr_save.on_click(pr_save_review)

pr_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Proposal Review & Approval</h2>"
    ),
    pr_refresh,
    pr_proposal,
    pr_load,
    widgets.HTML("<h3>Quality-Control Checklist</h3>"),
    widgets.VBox(list(pr_checks.values())),
    pr_notes,
    pr_save,
    pr_output
])

# Register in existing CRM menu



print("✓ No proposal records modified during setup.")


In [ ]:
# STEP 049 — Proposal document generation form

# Original section: PROPOSAL DOCUMENT GENERATION MANAGER
# NORTH STAR ANALYTICS CRM


pg_proposal = widgets.Dropdown(
    options=[("Select a proposal...", "")],
    description="Proposal:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="750px")
)

pg_refresh = widgets.Button(
    description="Refresh Proposals",
    icon="refresh"
)

pg_generate = widgets.Button(
    description="Generate Word Proposal",
    button_style="success",
    icon="file-text"
)

pg_output = widgets.Output()

pg_review_fields = [
    "Scope Confirmed",
    "Deliverables Confirmed",
    "Timeline Confirmed",
    "Pricing Confirmed",
    "Payment Terms Confirmed",
    "Exclusions Confirmed",
    "Add-Ons Confirmed",
    "Client Details Confirmed"
]

def pg_refresh_proposals(_=None):
    with pg_output:
        clear_output()

        try:
            options = [("Select a proposal...", "")]

            for record in proposals_sheet.get_all_records():
                proposal_id = str(
                    record.get("Proposal ID", "")
                ).strip()

                if proposal_id:
                    options.append((
                        f"{proposal_id} | "
                        f"${record.get('Total Price', '')} | "
                        f"{record.get('Status', '')}",
                        proposal_id
                    ))

            pg_proposal.options = options
            print(f"✓ {len(options) - 1} proposals loaded.")

        except Exception as error:
            print("Unable to load proposals:", error)

def pg_generate_document(_):
    with pg_output:
        clear_output()

        proposal_id = pg_proposal.value

        if not proposal_id:
            print("⚠ Select a proposal first.")
            return

        pg_generate.disabled = True

        try:
            # Retrieve the latest saved review
            review = show_proposal_review_status(proposal_id)

            if not review:
                print("⚠ No proposal review recorded.")
                return

            if str(review.get("Review Status", "")).strip() != "Client Ready":
                print("⚠ Proposal has not passed quality control.")
                return

            # Verify every individual approval
            missing = [
                field for field in pg_review_fields
                if str(review.get(field, "")).strip().lower()
                != "yes"
            ]

            if missing:
                print("⚠ Missing quality-control approvals:")
                for field in missing:
                    print(" •", field)
                return

            print("\n✓ All eight review checks confirmed.")
            print("Generating proposal document...\n")

            filepath = generate_proposal_document(proposal_id)

            print("\n✓ Word proposal ready for review.")
            print("Saved to:", filepath)
            print("No document was sent to a client.")

        except FileExistsError as error:
            print("⚠ A proposal document already exists.")
            print(error)
            print("The existing document was not overwritten.")

        except Exception as error:
            print("⚠ Document generation failed:")
            print(type(error).__name__, ":", error)

        finally:
            pg_generate.disabled = False

pg_refresh.on_click(pg_refresh_proposals)
pg_generate.on_click(pg_generate_document)

pg_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Proposal Documents</h2>"
    ),
    pg_refresh,
    pg_proposal,
    pg_generate,
    pg_output
])




print("✓ Quality-control safeguards enabled.")
print("✓ No documents generated during setup.")


In [ ]:
# STEP 050 — Proposal tracking and follow-up form

# Original section: PROPOSAL TRACKING MANAGER
# NORTH STAR ANALYTICS CRM


pt_style = {"description_width": "130px"}
pt_width = widgets.Layout(width="750px")

pt_proposal = widgets.Dropdown(
    options=[("Select a proposal...", "")],
    description="Proposal:",
    style=pt_style,
    layout=pt_width
)

pt_status = widgets.Dropdown(
    options=[
        "Draft", "Sent", "Under Review",
        "Accepted", "Declined", "Expired"
    ],
    description="New Status:",
    style=pt_style,
    layout=pt_width
)

pt_days = widgets.BoundedIntText(
    value=3,
    min=1,
    max=30,
    description="Business Days:",
    style=pt_style
)

pt_priority = widgets.Dropdown(
    options=["High", "Medium", "Low"],
    value="High",
    description="Priority:",
    style=pt_style
)

pt_contact = widgets.Dropdown(
    options=[("No contact selected", "")],
    description="Contact:",
    style=pt_style,
    layout=pt_width
)

pt_refresh = widgets.Button(
    description="Refresh Proposals",
    icon="refresh"
)

pt_update = widgets.Button(
    description="Update Proposal Status",
    button_style="info",
    icon="check"
)

pt_followup = widgets.Button(
    description="Schedule Proposal Follow-Up",
    button_style="success",
    icon="calendar"
)

pt_output = widgets.Output()

def pt_refresh_proposals(_=None):
    with pt_output:
        clear_output()

        try:
            records = proposals_sheet.get_all_records()
            options = [("Select a proposal...", "")]

            for r in records:
                pid = str(r.get("Proposal ID", "")).strip()

                if pid:
                    options.append((
                        f"{pid} | ${r.get('Total Price', '')} | "
                        f"{r.get('Status', '')}",
                        pid
                    ))

            pt_proposal.options = options
            print(f"✓ {len(options) - 1} proposals loaded.")

        except Exception as error:
            print("Unable to refresh proposals:", error)

def pt_load_contacts(change=None):
    proposal_id = pt_proposal.value

    pt_contact.options = [("No contact selected", "")]

    if not proposal_id:
        return

    proposals = proposals_sheet.get_all_records()

    proposal = next(
        (
            r for r in proposals
            if str(r.get("Proposal ID")) == proposal_id
        ),
        None
    )

    if not proposal:
        return

    company_id = str(proposal.get("Company ID", ""))

    options = [("No contact selected", "")]

    for contact in contacts_sheet.get_all_records():
        if str(contact.get("Company ID", "")) == company_id:
            name = (
                f"{contact.get('First Name', '')} "
                f"{contact.get('Last Name', '')}"
            ).strip()

            options.append((
                name or str(contact.get("Contact ID")),
                str(contact.get("Contact ID"))
            ))

    pt_contact.options = options

    current_status = str(proposal.get("Status", "Draft"))
    if current_status in pt_status.options:
        pt_status.value = current_status

def pt_update_status(_):
    proposal_id = pt_proposal.value
    new_status = pt_status.value
    with pt_output:
        clear_output(wait=True)
        if not proposal_id:
            print("⚠ Select a proposal.")
            return
        pt_update.disabled = True
        try:
            if new_status == "Sent":
                review = show_proposal_review_status(proposal_id)
                checks = [
                    "Scope Confirmed", "Deliverables Confirmed",
                    "Timeline Confirmed", "Pricing Confirmed",
                    "Payment Terms Confirmed", "Exclusions Confirmed",
                    "Add-Ons Confirmed", "Client Details Confirmed"
                ]
                if (not review
                    or str(review.get("Review Status", "")).strip() != "Client Ready"
                    or not all(str(review.get(k, "")).strip().lower() == "yes" for k in checks)):
                    print("⚠ All eight proposal quality checks must pass before marking Sent.")
                    return
            update_proposal_status(proposal_id, new_status)
            print("✓ Proposal status saved successfully.")
            print("Proposal:", proposal_id)
            print("Status:", new_status)
            print("No document or email was sent.")
            # Refresh dropdown labels without clearing this confirmation.
            records = proposals_sheet.get_all_records()
            options = [("Select a proposal...", "")]
            for record in records:
                pid = str(record.get("Proposal ID", "")).strip()
                if pid:
                    options.append((f"{pid} | ${record.get('Total Price', '')} | {record.get('Status', '')}", pid))
            pt_proposal.options = options
            if proposal_id in [v for _, v in options]:
                pt_proposal.value = proposal_id
        except Exception as error:
            print("⚠ Unable to update proposal:", error)
        finally:
            pt_update.disabled = False

def pt_schedule_followup(_):
    with pt_output:
        clear_output()

        proposal_id = pt_proposal.value

        if not proposal_id:
            print("⚠ Select a proposal.")
            return

        pt_followup.disabled = True

        try:
            proposal = next(
                (
                    r for r in proposals_sheet.get_all_records()
                    if str(r.get("Proposal ID")) == proposal_id
                ),
                None
            )

            if not proposal:
                print("⚠ Proposal not found.")
                return

            status = str(proposal.get("Status", "")).strip()

            if status not in ["Sent", "Under Review"]:
                print(
                    "⚠ Follow-ups can only be scheduled "
                    "for Sent or Under Review proposals."
                )
                return

            task_id = schedule_proposal_followup(
                proposal_id=proposal_id,
                business_days=pt_days.value,
                priority=pt_priority.value,
                contact_id=pt_contact.value
            )

            if task_id:
                print("✓ Follow-up added to CRM.")
            else:
                print("No duplicate follow-up created.")

        except Exception as error:
            print("Unable to schedule follow-up:", error)

        finally:
            pt_followup.disabled = False

pt_refresh.on_click(pt_refresh_proposals)
pt_proposal.observe(pt_load_contacts, names="value")
pt_update.on_click(pt_update_status)
pt_followup.on_click(pt_schedule_followup)

pt_form = widgets.VBox([
    widgets.HTML(
        "<h2>North Star Analytics — Proposal Tracking</h2>"
    ),
    pt_refresh,
    pt_proposal,
    pt_status,
    pt_update,
    widgets.HTML("<hr><h3>Proposal Follow-Up</h3>"),
    pt_days,
    pt_priority,
    pt_contact,
    pt_followup,
    pt_output
])




print("✓ No proposal statuses changed during setup.")


In [ ]:
# STEP 051 — Accepted proposal project conversion function

# Original section: SAFE PROPOSAL-TO-PROJECT CONVERSION
# Defines a function only. Does not create a project.


def create_project_from_proposal(
    proposal_id,
    project_name,
    start_date,
    target_completion_date,
    hours_estimated=0,
    project_folder_link=""
):
    """Convert an accepted proposal into a project safely."""

    proposals = proposals_sheet.get_all_records()

    proposal = next(
        (
            p for p in proposals
            if str(p.get("Proposal ID", "")).strip()
            == str(proposal_id).strip()
        ),
        None
    )

    if proposal is None:
        raise ValueError("Proposal not found.")

    if str(proposal.get("Status", "")).strip() != "Accepted":
        raise ValueError(
            "Only accepted proposals can become projects."
        )

    company_id = str(proposal.get("Company ID", "")).strip()
    opportunity_id = str(
        proposal.get("Opportunity ID", "")
    ).strip()

    if not company_id or not opportunity_id:
        raise ValueError(
            "Proposal must have a company and opportunity."
        )

    existing_projects = projects_sheet.get_all_records()

    for project in existing_projects:
        if str(project.get("Opportunity ID", "")).strip() == opportunity_id:
            raise ValueError(
                "A project already exists for this opportunity: "
                + str(project.get("Project ID", ""))
            )

    start = date.fromisoformat(str(start_date))
    target = date.fromisoformat(str(target_completion_date))

    if target < start:
        raise ValueError(
            "Target completion date cannot precede start date."
        )

    if not str(project_name).strip():
        raise ValueError("Project name is required.")

    if float(hours_estimated) < 0:
        raise ValueError("Estimated hours cannot be negative.")

    contract_value = proposal.get("Total Price", "")

    if contract_value in ("", None):
        raise ValueError("Proposal contract value is missing.")

    service_type = str(
        proposal.get("Service Package", "")
    ).strip()

    deliverables = str(proposal.get("Scope", "")).strip()

    return create_project(
        company_id=company_id,
        project_name=str(project_name).strip(),
        service_type=service_type,
        start_date=start.isoformat(),
        target_completion_date=target.isoformat(),
        contract_value=contract_value,
        opportunity_id=opportunity_id,
        hours_estimated=float(hours_estimated),
        deliverables=deliverables,
        project_folder_link=project_folder_link,
        notes=f"Created from accepted proposal {proposal_id}"
    )

print("✓ Safe proposal-to-project function loaded.")
print("✓ Accepted proposal required.")
print("✓ Duplicate opportunity projects blocked.")
print("✓ Existing project records unchanged.")


In [ ]:
# STEP 052 — Project launch form

# Original section: PROJECT LAUNCH MANAGER


pl_proposal = widgets.Dropdown(
    options=[("Select an accepted proposal...", "")],
    description="Proposal:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)

pl_details = widgets.HTML()

pl_name = widgets.Text(
    description="Project Name:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)

pl_start = widgets.DatePicker(
    description="Start Date:",
    value=date.today(),
    style={"description_width": "initial"}
)

pl_target = widgets.DatePicker(
    description="Target Completion:",
    value=date.today() + timedelta(days=30),
    style={"description_width": "initial"}
)

pl_hours = widgets.BoundedFloatText(
    value=0,
    min=0,
    max=100000,
    description="Estimated Hours:",
    style={"description_width": "initial"}
)

pl_folder = widgets.Text(
    description="Project Folder Link:",
    placeholder="Optional Google Drive folder link",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)

pl_refresh = widgets.Button(
    description="Refresh Accepted Proposals",
    button_style="info"
)

pl_launch = widgets.Button(
    description="Launch Project",
    button_style="success",
    icon="play"
)

pl_output = widgets.Output()


def pl_refresh_proposals(_=None):
    proposals = proposals_sheet.get_all_records()
    projects = projects_sheet.get_all_records()

    used_opportunities = {
        str(p.get("Opportunity ID", "")).strip()
        for p in projects
        if str(p.get("Opportunity ID", "")).strip()
    }

    options = [("Select an accepted proposal...", "")]

    for proposal in proposals:
        if str(proposal.get("Status", "")).strip() != "Accepted":
            continue

        opportunity_id = str(
            proposal.get("Opportunity ID", "")
        ).strip()

        if not opportunity_id or opportunity_id in used_opportunities:
            continue

        proposal_id = str(proposal.get("Proposal ID", "")).strip()
        company_id = str(proposal.get("Company ID", "")).strip()

        label = f"{proposal_id} | {company_id}"

        options.append((label, proposal_id))

    pl_proposal.options = options

    with pl_output:
        clear_output()
        print(f"✓ {len(options) - 1} eligible accepted proposal(s).")


def pl_show_details(change=None):
    proposal_id = pl_proposal.value

    if not proposal_id:
        pl_details.value = (
            "<i>Select an accepted proposal to view its details.</i>"
        )
        return

    proposal = next(
        (
            p for p in proposals_sheet.get_all_records()
            if str(p.get("Proposal ID", "")).strip() == proposal_id
        ),
        None
    )

    if proposal is None:
        pl_details.value = "<b>Proposal not found.</b>"
        return

    from html import escape

    company_id = escape(str(proposal.get("Company ID", "")))
    service = escape(str(proposal.get("Service Package", "")))
    value = escape(str(proposal.get("Total Price", "")))

    pl_details.value = (
        f"<b>Company:</b> {company_id}<br>"
        f"<b>Service Package:</b> {service}<br>"
        f"<b>Contract Value:</b> ${value}"
    )


def pl_launch_project(_):
    with pl_output:
        clear_output()

        if not pl_proposal.value:
            print("⚠ Select an accepted proposal.")
            return

        if not pl_name.value.strip():
            print("⚠ Enter a project name.")
            return

        if pl_start.value is None or pl_target.value is None:
            print("⚠ Both project dates are required.")
            return

        pl_launch.disabled = True

        try:
            project_id = create_project_from_proposal(
                proposal_id=pl_proposal.value,
                project_name=pl_name.value.strip(),
                start_date=pl_start.value.isoformat(),
                target_completion_date=pl_target.value.isoformat(),
                hours_estimated=pl_hours.value,
                project_folder_link=pl_folder.value.strip()
            )

            print("\n✓ Project launched successfully!")
            print("Project ID:", project_id)
            print("Status: Not Started")

            pl_refresh_proposals()

        except Exception as error:
            print("⚠ Project launch failed:", error)

        finally:
            pl_launch.disabled = False


pl_proposal.observe(pl_show_details, names="value")
pl_refresh.on_click(pl_refresh_proposals)
pl_launch.on_click(pl_launch_project)

pl_form = widgets.VBox([
    widgets.HTML("<h3>Project Launch Manager</h3>"),
    pl_refresh,
    pl_proposal,
    pl_details,
    pl_name,
    pl_start,
    pl_target,
    pl_hours,
    pl_folder,
    pl_launch,
    pl_output
])

def open_project_launch_manager():
    pl_refresh_proposals()
    display(pl_form)


print("✓ Only accepted proposals are eligible.")
print("✓ Existing projects and tasks unchanged.")


## Future development (comment-only)

The following are planned features, not implemented functions. They are intentionally placed **above the final launcher** so the launcher remains the last executable cell.

In [ ]:
# STEP 053 — FUTURE — Project task status management

# TODO: Add a task status editor, completion dates, and audit checks.
# TODO: Implement, review, and test before enabling in the launcher.


In [ ]:
# STEP 054 — FUTURE — Project progress and billable hours

# TODO: Add tracked hours, task summaries, and project profitability calculations.
# TODO: Implement, review, and test before enabling in the launcher.


In [ ]:
# STEP 055 — FUTURE — Continuing service and renewal workflow

# TODO: Add recurring client service forms and renewal follow-up controls.
# TODO: Implement, review, and test before enabling in the launcher.


In [ ]:
# STEP 056 — FUTURE — End-to-end testing and release readiness

# TODO: Test with a fictional workbook; verify permissions, duplicates, data validation, and no unexpected writes.
# TODO: Implement, review, and test before enabling in the launcher.


In [ ]:
# STEP 057 — LAUNCH CRM — single interactive menu (run last)

# This is the only menu registration and display cell.
# Earlier cells define functions/forms without opening duplicate dashboards.
crm_actions = {
    "Daily Command Center": north_star_daily_command_center,
    "Sales Activity Dashboard": daily_sales_command_center,
    "Daily Outreach Report": show_daily_outreach_report,
    "Weekly Outreach Report": show_weekly_outreach_report,
    "Priority Opportunities": show_priority_opportunities,
    "Outstanding Invoices": show_outstanding_invoices,
    "Financial Summary": show_financial_summary,
    "Recurring Revenue": show_recurring_revenue,
    "Upcoming Renewals": show_upcoming_renewals,
    "Add New Company": lambda: display(company_form),
    "Add New Contact": lambda: display(contact_form),
    "Log Outreach Activity": lambda: display(outreach_form),
    "Schedule Follow-Up": lambda: display(fu_form),
    "View & Complete Follow-Ups": lambda: (refresh_task_list(), display(task_management_form)),
    "Add New Opportunity": lambda: display(opp_form),
    "Update Opportunity Stage": lambda: (stage_load_opportunities(), display(stage_form)),
    "Score Opportunity": lambda: (score_load_opportunities(), display(score_form)),
    "Create Discovery Interview": lambda: (di_refresh_lists(), display(di_form)),
    "Record Discovery Responses": lambda: (ir_load_interviews(), display(ir_form)),
    "Discovery Findings & Recommendations": lambda: (df_load_interviews(), display(df_form)),
    "Proposal Scope Draft Manager": lambda: (ps_refresh_lists(), display(ps_form)),
    "Proposal Review & Approval": lambda: (pr_refresh_proposals(), display(pr_form)),
    "Generate Proposal Document": lambda: (pg_refresh_proposals(), display(pg_form)),
    "Proposal Tracking Manager": lambda: (pt_refresh_proposals(), display(pt_form)),
    "Project Launch Manager": open_project_launch_manager,
}
crm_menu = widgets.Dropdown(
    options=list(crm_actions),
    description="CRM Action:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="470px")
)
crm_open_button = widgets.Button(description="Open Selected Action", button_style="primary", icon="folder-open")
crm_menu_output = widgets.Output()
def open_selected_crm_action(_):
    with crm_menu_output:
        clear_output(wait=True)
        try:
            crm_actions[crm_menu.value]()
        except Exception as error:
            print(f"Unable to open {crm_menu.value}: {error}")
crm_open_button.on_click(open_selected_crm_action)
display(widgets.VBox([
    widgets.HTML("<h2>North Star Analytics CRM</h2><p>Select an action to begin.</p>"),
    crm_menu, crm_open_button, crm_menu_output
]))
print(f"CRM launcher ready: {len(crm_actions)} actions.")
